# YOLO26m · combo 단일 holdout NMS-free(one2one 헤드) + AI Hub 외부 데이터 · 모자이크 + bbox copy-paste · 02_eda 검수 선별 · 각인 OCR 재분류 (CUDA·MPS)

`exp_yolo26m_nms_free.ipynb`를 복사해 **train에만 AI Hub 외부 사진을 더한** 실험입니다.
외부 사진은 `02_eda.ipynb` §5~8과 같은 방법·seed로 고르고, 팀원 yolo11m AI Hub 노트북
(`yolo11m_100_aihub_not_background.ipynb`)과 같이 §8 문제 사진을 뺍니다(2-2절).
validation·평가·속도·제출 csv는 **대회 사진만** 씁니다.
`USE_AIHUB = False`로 바꾸면 원본 NMS-free 노트북과 같은 데이터로 학습합니다.

`exp_yolo26m_nms_free_aihub_mosaic0.ipynb`를 복사해 **모자이크를 다시 켜고(`MOSAIC = 1.0`)
bbox copy-paste 증강을 더한** 실험입니다. 그 밖의 설정은 `exp_yolo26m_nms_free_aihub.ipynb`와 같습니다.

ultralytics의 `copy_paste`는 segmentation 라벨이 있어야 동작해서, bbox만 있는 우리 데이터에서는 아무것도
하지 않습니다(`AUG_PRESETS`에서도 0). 그래서 3-2절에서 학습 사진(대회 train + AI Hub)의
**정답 bbox를 그대로 잘라**, 학습 사진을 읽을 때 **기존 정답과 겹치지 않는 배경 위치에 붙이고 라벨을 더합니다**.
외부 전용 클래스를 포함한 전체 클래스에서 균등하게 고릅니다. 자세한 방식은 3-2절에 있습니다.
`COPY_PASTE_PROB = 0.0`이면 AI Hub 노트북과 같은 설정이 됩니다.

`exp_yolo26m_nms_free_aihub_copypaste.ipynb`를 복사해 **AI Hub 외부 사진을 `02_eda.ipynb`(exp/aihub_eda 브랜치)의
선별·검수 결과로 바꾼** 실험입니다. 노트북 안에서 다시 뽑지 않고 `data/processed/02_eda_selection.csv`
(선별 bbox, 02_eda §10~11 사람 검수의 승인 보정 반영)를 그대로 쓰고, `02_eda_review_decisions.csv`의
승인 보정을 사진 전체 라벨에 적용합니다(2-2절). 모자이크·copy-paste·하이퍼파라미터는 copypaste 노트북과 같습니다.

`exp_yolo26m_nms_free_aihub_copypaste_eda.ipynb`를 복사해 **알약 각인(식별문자) OCR 모델을 함께 학습하고
탐지 결과의 클래스를 OCR로 다시 고르는** 실험입니다(5-2절). YOLO 학습은 eda 노트북과 같고, YOLO 학습이 끝나면
같은 학습 사진의 정답 bbox crop으로 작은 OCR 모델(CRNN + CTC)을 학습합니다. 정답 텍스트는 JSON의
`print_front`·`print_back` 중 그 사진에서 보이는 면(`drug_dir`)입니다. 추론 때는 탐지 bbox마다 모든 클래스의
각인이 나올 확률을 OCR로 계산해, OCR이 다른 클래스를 강하게 가리킬 때만 클래스를 바꿉니다.
validation mAP75-95를 OCR 전후로 비교하고, 제출 csv는 OCR 전·후 두 개를 만듭니다(8·8-2절).
`USE_OCR = False`면 eda 노트북과 같습니다.

프로젝트의 **uv Python 3.12 가상환경을 커널로 선택**한 뒤 위에서 아래로 실행합니다.
데이터 준비 셀은 준비 파일을 만들고, **학습 셀은 단일 train/validation으로 한 번 학습**합니다.
장치는 `get_device()`로 CUDA > MPS 순으로 고르며 **CUDA·MPS 모두 본 학습**을 할 수 있습니다.
동작 확인만 하려면 설정 셀에서 `SMOKE = True`(1 epoch, 학습 데이터 10%)로 바꿉니다.
이미 학습한 실험은 설정 셀의 `REUSE_DIR`에 그 결과 폴더를 넣으면 학습을 건너뛰고
그 best.pt로 평가·그림·속도·제출 csv만 다시 만듭니다. 학습이 중간에 끊긴 실험이면 이어서 학습합니다.
작성된 노트북에는 실행 결과를 저장하지 않았습니다.

**Colab에서 실행하기** (0절이 Colab에서만 준비합니다. Mac에서는 0절이 아무것도 하지 않습니다)

1. 이 브랜치(`exp/yolo26m_nms_free_aihub_copypaste_eda_ocr`)의 노트북을 Colab에서 열고 **런타임 → 런타임 유형 변경**에서
   A100 또는 L4 GPU를 고릅니다.
2. 위에서 아래로 실행합니다. 0절이 Drive 마운트, 저장소 받기, 패키지 설치, 대회 데이터 받기를 합니다.
   대회 데이터 링크(`DATA_URL`)와 AI Hub 링크(`AIHUB_URL`)는 입력창이 뜨면 붙여 넣습니다(화면에 표시되지 않음).
3. 결과 폴더는 Drive의 `MyDrive/pillbbox_runs/<실험 이름>`이라 런타임이 끊겨도 `best.pt`·`last.pt`가 남습니다.
   학습 입력(`prepared`)과 외부 데이터는 런타임 디스크에 둡니다(약 48GB, 받고 푸는 데 수십 분).
   **AI Hub 공유 폴더를 내 드라이브에 바로가기로 추가해 두세요**(폴더 메뉴 → 정리 → 바로가기 추가 →
   내 드라이브). 바로가기(`MyDrive/aihub_pill`)가 있으면 링크로 받지 않고 거기서 복사해서, 큰 공유 파일에
   걸리는 Drive 다운로드 제한("Too many users have viewed or downloaded this file recently")을 피합니다.
4. **끊기면 이어 학습**: 새 런타임에서 설정 셀의 `REUSE_DIR`에 그 실험 폴더 이름을 넣고 처음부터 다시 실행합니다.
   데이터는 같은 seed로 다시 만들고(이전과 같은지 검사), 학습은 `train/weights/last.pt`에서 이어 갑니다.
   조기 중단 카운트(최고 점수와 그 에폭)도 이어서 셉니다.
5. **다 끝나면 런타임 삭제**: 제출 csv까지 만들면 마지막 셀(9절)이 Drive에 결과 쓰기를 마치고
   Colab 런타임을 삭제해 GPU 요금이 더 나가지 않게 합니다. 끄지 않으려면 설정 셀의
   `COLAB_RELEASE_RUNTIME = False`로 바꿉니다. 앞 셀에서 에러가 나면 런타임이 남아 있으니 직접 끊습니다.

| 항목 | 설정 |
|---|---|
| 모델 / 전이학습 | yolo26m.pt / full fine tuning (freeze=None) |
| 원본 | train 232장, test 842장. test는 학습·검증에서 제외 |
| 외부 데이터 | AI Hub 경구약제 조합(TL_2·TS_2 제외). **02_eda 선별 csv**(공통 54종을 157 bbox까지 보충, seed 42)의 bbox가 든 사진을 통째로 **train에만** 추가 |
| 외부 라벨 보정 | **02_eda §10~11 사람 검수 결과 적용**(`02_eda_review_decisions.csv`): `승인`은 새 bbox, `원본 유지`는 그대로, `이미지 제외`는 사진을 뺌 |
| 외부 전용 알약 | 대회 56종에 없는 약은 **새 클래스**로 학습하고 평가 예측에서는 뺀다. 제출에는 약 코드를 category_id로 넣는다(`SUBMIT_EXTRA_CLASSES`) |
| 외부 사진 제외 | 02_eda 검수의 `이미지 제외`(겹침·라벨 누락·잘림). 그 뒤 안전 검사로 대회 사진과 같은 이름, 02_eda §8 문제 사진(라벨 누락·파일명에 없는 라벨·bbox 겹침·좌표 오류·JSON과 실제 크기 다름), 대회 사진과 같은 약 조합 |
| 라벨 | 현재 파이프라인의 v1 보정 라벨. 원본 JSON을 수정하지 않음 |
| 분할 | combo 단위 단일 holdout, 희귀 조합 train 고정, 교차검증 없음, seed=42 |
| 희귀 알약 | v1 전체 bbox가 5개 이하인 클래스가 든 **조합 전체를 train에 고정** |
| 비율 | 고정분을 제외한 조합을 약 8:2로 분할. 전체 이미지를 185/47로 강제하지 않음 |
| epochs / patience | 최대 100 / mAP75–95가 10에폭 연속 개선되지 않으면 중단 |
| optimizer / lr0 | MuSGD / 0.001 |
| batch / imgsz | 16 / 640 |
| 증강 | 기존 AUG_PRESETS["on"]: 회전 우선, 반전 끔, 약한 hue. **모자이크 켬(`MOSAIC = 1.0`, 마지막 10에폭 끔)** |
| bbox copy-paste | **모자이크 타일마다 확률 0.5로 1~2개. crop은 학습 사진의 정답 bbox, 전체 클래스(외부 전용 포함) 균등, 마지막 10에폭 끔** |
| 각인 OCR | **YOLO 학습 뒤 따로 학습.** 학습 사진의 정답 bbox crop(128px, 임의 회전·흔들기), CRNN + CTC, AdamW lr 0.001, 최대 30에폭 / 검증 각인 분류 정확도가 10에폭 오르지 않으면 중단 |
| OCR 재분류 | 8방향 회전 중 최고 CTC 확률로 클래스별 OCR 확률을 구해, OCR 최고 클래스 ≥ 0.9이고 YOLO 클래스 ≤ 0.05일 때만 클래스를 바꿈. 박스·점수는 YOLO 그대로 |
| 평가 / 그림 신뢰도 | mAP 평가 0.001 / 혼동행렬 0.001·0.25 / 이미지 표시 0.25 |
| 후처리 | NMS 없음(YOLO26 기본 one2one 헤드, end2end), max_det=4, 평가에서 외부 전용 클래스 예측 제거(제출은 포함) |
| 성능 | 단일 holdout의 mAP@[0.75:0.95], Precision, Recall, 추론속도와 꺾은선 그래프 |
| 저장 | best.pt, 에폭별 성능·손실 그래프, 이미지별 추론시간 그래프, AP·혼동행렬·GT/예측, OCR 가중치·검증 재분류 표, 대회 제출 csv(OCR 전·후) |

**검증에서 역전파나 optimizer.step()을 하지 않습니다.**
eval 모드·inference_mode를 적용하고 검증 전후 가중치·버퍼가 같은지도 검사합니다.

실행 시 산출물은 runs/(Colab은 Drive의 MyDrive/pillbbox_runs/) 아래 새 실험 폴더에 모읍니다. 기존 raw·보정 CSV·공용 산출물은
덮어쓰지 않습니다. YOLO 라벨·분할 목록·data.yaml은 data_process.data_main.run()이 만듭니다.

## 0. Colab 준비 (Colab에서만 실행)

Colab이면 Drive를 마운트하고, `/content/pill-bbox`에 이 브랜치를 받고, `uv.lock` 버전으로 패키지를 설치하고,
대회 데이터가 없으면 받습니다. 이미 받은 저장소·데이터는 그대로 씁니다. Mac(로컬)에서는 아무것도 하지 않습니다.
패키지 설치 뒤 커널을 재시작하라는 에러가 나오면 재시작하고 이 셀부터 다시 실행합니다.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import urllib.request
from getpass import getpass
from importlib import invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

IN_COLAB = bool(os.getenv("COLAB_RELEASE_TAG"))
COLAB_REPO_URL = "https://github.com/def-uk/pill-bbox.git"
COLAB_BRANCH = (
    "exp/yolo26m_nms_free_aihub_copypaste_eda_ocr"  # origin에 push된 브랜치만 받을 수 있다
)
COLAB_REPO_DIR = Path("/content/pill-bbox")
# 실험 폴더(best.pt·last.pt·그림·제출 csv)는 Drive에 둬서 런타임이 끊겨도 남긴다.
COLAB_OUTPUT_ROOT = Path("/content/drive/MyDrive/pillbbox_runs")
# 학습 입력(prepared)은 런타임 디스크에 둔다. Drive에는 이미지 링크를 만들 수 없어
# 복사되고 읽기도 느리다. data_main이 저장소 기준 경로로 출력하므로 저장소 안(runs/는
# 커밋 안 됨)에 둔다.
COLAB_PREPARED_ROOT = COLAB_REPO_DIR / "runs" / "prepared"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not COLAB_REPO_DIR.exists():
        print(f"저장소를 받습니다: {COLAB_REPO_DIR}, branch={COLAB_BRANCH}")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                COLAB_BRANCH,
                COLAB_REPO_URL,
                str(COLAB_REPO_DIR),
            ],
            check=True,
        )
    os.chdir(COLAB_REPO_DIR)
    if str(COLAB_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(COLAB_REPO_DIR))
    commit = subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True, check=True
    )
    print(f"저장소: {COLAB_REPO_DIR} ({commit.stdout.strip()})")
    if sys.version_info < (3, 12):  # noqa: UP036 - Colab 커널의 실제 버전을 검사한다.
        raise RuntimeError(
            "이 프로젝트는 Python 3.12 이상이 필요합니다. Colab 런타임을 확인하세요."
        )

    # Colab 커널 자체에 uv.lock 버전의 패키지를 설치한다(yolo11m AI Hub 노트북과 같은 방법).
    uv = shutil.which("uv")
    if uv is None:
        uv_dir = Path("/tmp/colab-uv")
        uv = str(uv_dir / "uv")
        if not Path(uv).is_file():
            with urllib.request.urlopen("https://astral.sh/uv/install.sh", timeout=60) as response:
                installer = response.read().decode("utf-8")
            subprocess.run(
                ["sh"],
                input=installer,
                text=True,
                check=True,
                env={**os.environ, "UV_UNMANAGED_INSTALL": str(uv_dir)},
            )
    requirements = Path("/tmp/colab-requirements.txt")
    subprocess.run(
        [uv, "export", "--frozen", "--no-dev", "--no-hashes", "--no-emit-project"]
        + ["-o", str(requirements)],
        check=True,
    )
    # uv.lock은 Linux에서 torch·torchvision을 CUDA 13.0 빌드(+cu130)로 고정한다. 이 빌드는
    # GPU 드라이버가 CUDA 13.0 이상을 지원해야 돌므로, 드라이버가 낮으면 Colab에 깔린 torch를 쓴다.
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    found = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi.stdout)
    driver_cuda = (int(found[1]), int(found[2])) if found else (0, 0)
    print(f"GPU 드라이버가 지원하는 CUDA: {driver_cuda[0]}.{driver_cuda[1]}")
    if driver_cuda < (13, 0):
        lines = requirements.read_text(encoding="utf-8").splitlines()
        keep = [line for line in lines if not line.startswith(("torch==", "torchvision=="))]
        requirements.write_text("\n".join(keep) + "\n", encoding="utf-8")
        print(
            "드라이버가 CUDA 13.0 미만이라 torch·torchvision은 Colab에 깔린 버전을 그대로 씁니다."
        )
    install = subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "-r", str(requirements)]
        + ["--extra-index-url", "https://download.pytorch.org/whl/cu130"]
        + ["--index-strategy", "unsafe-best-match", "--break-system-packages"],
        capture_output=True,
        text=True,
        check=False,
    )
    if install.returncode:
        # Colab은 하위 프로세스 출력을 셀에 보여 주지 않으므로 에러를 직접 출력한다.
        print(install.stderr[-3000:])
        raise RuntimeError(
            f"패키지 설치 실패(종료 코드 {install.returncode}). 위 메시지를 확인하세요."
        )
    print(install.stderr.strip().splitlines()[-1] if install.stderr.strip() else "패키지 설치 완료")
    invalidate_caches()
    # Colab이 미리 불러온 패키지는 설치만으로 메모리의 버전이 바뀌지 않는다.
    distributions = {
        "numpy": "numpy",
        "pandas": "pandas",
        "torch": "torch",
        "ultralytics": "ultralytics",
        "matplotlib": "matplotlib",
        "PIL": "pillow",
    }
    stale = []
    for module_name, distribution in distributions.items():
        module = sys.modules.get(module_name)
        if module is None or not hasattr(module, "__version__"):
            continue
        try:
            if str(module.__version__) != version(distribution):
                stale.append(module_name)
        except PackageNotFoundError:
            continue
    if stale:
        raise RuntimeError(
            f"패키지 설치가 끝났지만 메모리에 이전 버전이 남아 있습니다: {stale}. "
            "커널을 재시작한 뒤 이 셀부터 다시 실행하세요."
        )

    # 대회 원본이 없으면 받는다. 링크는 저장소에 남기지 않으려고 입력창으로 받는다.
    raw_root = COLAB_REPO_DIR / "data" / "raw"
    if not any(
        (path / "train_images").is_dir()
        for path in (raw_root, raw_root / "sprint_ai_project1_data")
    ):
        from data_process.download_data import download_data

        # 링크 대신 마운트한 Drive의 zip 경로(/content/drive/MyDrive/....zip)를 넣으면
        # gdown을 거치지 않아 Drive 다운로드 횟수 제한에 걸리지 않는다.
        download_data(
            getpass("대회 데이터 Drive 링크 또는 Drive zip 경로 (화면에 표시되지 않음): ").strip()
        )
print(f"Colab={IN_COLAB}")

In [ ]:
import os
import sys
from pathlib import Path

# 루트 또는 notebooks에서 실행해도 프로젝트 import 경로를 찾는다.
PROJECT_PATH = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "config.py").is_file()
    ),
    None,
)
if PROJECT_PATH is None:
    raise RuntimeError("pill-bbox 프로젝트 안에서 노트북을 실행하세요.")
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_AUTOINSTALL"] = "false"
# MPS에 없는 연산은 CPU로 넘겨 실행한다. torch를 import하기 전에 정해야 한다.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

In [ ]:
import gc
import json
import math
import multiprocessing
import platform
import random
import shutil
import unicodedata
from copy import copy
from dataclasses import asdict
from datetime import datetime
from getpass import getpass
from multiprocessing.pool import ThreadPool
from time import perf_counter
from unittest.mock import patch
from zoneinfo import ZoneInfo

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from dotenv import load_dotenv
from IPython import get_ipython
from IPython.display import Image, display
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import GroupShuffleSplit
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT
from ultralytics.data.dataset import YOLODataset
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils import LOGGER, NUM_THREADS
from ultralytics.utils.metrics import ConfusionMatrix, ap_per_class
from ultralytics.utils.torch_utils import EarlyStopping, init_seeds

from data_process import data_yaml
from data_process.aihub_data import (
    COMBO_KEYWORD,
    download_drive_folder,
    extract_zips,
    find_combo_zips,
    list_combo_images,
    load_combo_annotations,
    zip_key,
)
from data_process.data_processing import (
    bbox_flags,
    find_box_pairs,
    list_images,
    load_annotations,
    read_image_headers,
)
from data_process.make_labels import (
    LABEL_VERSIONS,
    file_sha256,
    read_yolo_labels,
    write_yolo_labels,
)
from data_process.move_files import link_images
from data_process.split import FOLD_COLS, RARE_MAX, SplitSpec, rare_classes
from models.evaluate import CONFUSION_PAIRS, MAIN_METRIC, ap75_95, map75_95, size_group
from models.train import AUG_PRESETS, MAIN_METRIC_KEY, Map7595Trainer, is_finished
from src.config import PROJECT_ROOT, RUNS_DIR, get_device

## 1. 설정과 장치

full fine tuning을 사용합니다. 그 밖의 미지정 학습 인자는 Ultralytics 기본값이며 실제 인자를
train/args.yaml에 남깁니다. lr0는 시작 학습률입니다. warmup·스케줄과 MuSGD의 일부
head 그룹 3배 배율은 유지하고 실제 그룹별 학습률을 기록합니다.

`NMS_ENABLED=True`면 one2many 헤드 + 클래스별 NMS(IoU=0.5), `False`면 YOLO26 기본인
one2one 헤드(NMS 없음, end2end)로 학습 중 검증·최종 평가·속도 측정을 모두 합니다.
`MOSAIC`은 모자이크 적용 확률로 `AUG_PRESETS` 밖에서 덮어씁니다(1.0 = ultralytics 기본값, 0.0 끔).

`COPY_PASTE_*`는 bbox copy-paste 설정입니다(3-2절, 학습 데이터셋에만 적용).
copy-paste 데이터셋은 이 노트북에서 정의한 클래스라 Mac·Windows처럼 worker를 spawn 방식으로 띄우는
환경에서는 worker가 불러오지 못합니다. 그때는 학습을 `workers=0`(메인 프로세스에서 데이터 읽기)으로
돌려 느려집니다. Colab(Linux, fork 방식)은 `WORKERS` 그대로 씁니다(`TRAIN_WORKERS`).

`AIHUB_*`는 외부 데이터 설정입니다. 외부 전용 알약을 새 클래스로 학습하므로 모델 클래스 수는
대회 56종보다 많습니다. 학습 중 검증·최종 평가에서는 외부 전용 클래스 예측을 지우고
대회 56종만 남깁니다(max_det=4 안에서 지우므로 이미지당 박스가 4개보다 적을 수 있습니다).
제출은 `SUBMIT_EXTRA_CLASSES`가 True면 외부 전용 클래스 예측도 약 코드를 category_id로 넣습니다
(test에는 대회 56종 밖의 알약도 있습니다).

`OCR_*`는 5-2절 각인 OCR 설정입니다. YOLO 학습에는 영향이 없고, YOLO 학습이 끝난 뒤 OCR을 따로 학습해
탐지 클래스를 다시 고릅니다. OCR 학습이 끝난 실험은 `ocr/completed.json`의 설정이 같으면 다시 학습하지 않습니다.

최종 평가·시각화의 기반 예측·속도 측정은 FP32를 쓰며, 학습 AMP는 기본 True이며 MPS에서는 ultralytics가 자동으로 끕니다.

In [ ]:
MODEL_NAME = "yolo26m.pt"
LABEL_VERSION = "v1"
SEED = 42
VAL_FRACTION = 0.2  # 희귀 조합을 제외한 나머지 조합의 validation 비율.
PIPELINE_PARTITION_ID = 0  # 공용 파일 형식에 필요한 식별자이며 교차검증 번호가 아니다.
SPLIT_NAME = f"combo_{LABEL_VERSION}_s{SEED}"
EPOCHS = 100
PATIENCE = 10
LR0 = 0.001
OPTIMIZER = "MuSGD"
BATCH = 16
IMGSZ = 640
FREEZE = None
AUG = "on"
# 모자이크 적용 확률. 1.0은 ultralytics 기본값(AI Hub 노트북), 0.0은 끔(모자이크 끔 노트북).
# 프리셋 밖에서 넣는다(팀원 yolo11m 모자이크 실험과 같은 방식).
MOSAIC = 1.0
AUGMENTATION = {**AUG_PRESETS[AUG], "mosaic": MOSAIC}
# bbox copy-paste (3-2절). ultralytics copy_paste는 segmentation 라벨이 없으면
# 동작하지 않으므로 AUG_PRESETS의 0을 그대로 두고 학습 데이터셋에서 따로 붙인다.
# crop은 학습 사진의 정답 bbox이고 외부 전용 클래스까지 전체 클래스 균등으로 고른다.
COPY_PASTE_PROB = 0.5  # 사진(모자이크 타일)마다 붙일 확률. 0.0이면 끔
COPY_PASTE_MAX = 2  # 한 번에 붙일 최대 개수(1~이 값에서 고름)
EVAL_CONF = 0.001
CM_CONFS = (0.001, 0.25)
VIS_CONF = 0.25
CM_MATCH_IOU = 0.45  # 라이브러리 기본 정답 매칭 기준. NMS IoU와 구분한다.
NMS_IOU = 0.5
NMS_ENABLED = False
MAX_DET = 4
GPU_INDEX = 0  # CUDA에서만 사용한다.
# 이미지 읽기·증강을 하는 프로세스 수. 0이면 학습과 같은 프로세스에서 차례로 처리해
# AI Hub를 더한 학습(사진 29배)에서 병목이 된다. Windows에서 멈추면 0으로 되돌린다.
WORKERS = 8
# 3-2절 copy-paste 데이터셋은 노트북에서 정의한 클래스라 spawn 방식 worker(Mac·Windows)가
# 불러올 수 없다. 그때는 학습 데이터를 메인 프로세스에서 읽는다(workers=0, 느려짐).
TRAIN_WORKERS = (
    WORKERS if COPY_PASTE_PROB == 0 or multiprocessing.get_start_method() == "fork" else 0
)
VERIFY_VALIDATION_WEIGHTS = True
BENCHMARK_WARMUP = 5
BENCHMARK_ROUNDS = 3
# 장치는 get_device()로 고른다(CUDA > MPS). CUDA·MPS 모두 본 학습을 할 수 있다.
DEVICE_TYPE = get_device()
SMOKE = True  # True: 동작 확인(1 epoch, 학습 데이터 10%). 본 학습은 False
SMOKE_EPOCHS = 1
SMOKE_FRACTION = 0.1
TRAIN_EPOCHS = SMOKE_EPOCHS if SMOKE else EPOCHS
TRAIN_FRACTION = SMOKE_FRACTION if SMOKE else 1.0
# Colab에서 제출 csv까지 만들면 런타임을 삭제해 GPU 요금(컴퓨팅 단위)을 아낀다(9절).
# smoke 실행은 끄지 않는다.
COLAB_RELEASE_RUNTIME = True

# AI Hub 외부 데이터 (02_eda 선별·검수 csv). train에만 더하고 validation은 대회 사진 그대로 둔다.
USE_AIHUB = True  # False: 원본 NMS-free 노트북과 같은 대회 데이터만 학습
# 02_eda(exp/aihub_eda 브랜치)가 만든 팀 공유 csv. 선별 bbox는 §10~11 검수의 승인 보정을
# 반영했고, 검수에서 '이미지 제외'한 사진은 빠져 있다. 노트북은 이 목록을 다시 뽑지 않는다.
AIHUB_SELECTION_CSV = PROJECT_ROOT / "data" / "processed" / "02_eda_selection.csv"
AIHUB_REVIEW_CSV = PROJECT_ROOT / "data" / "processed" / "02_eda_review_decisions.csv"
# 02_eda §5의 클래스당 bbox 목표(대회 최다 클래스 수 157). 선별은 csv를 그대로 쓰고
# 이 값은 목표를 넘는 클래스가 없는지 검사하고 그래프에 표시하는 데만 쓴다.
AIHUB_TARGET_BBOX = 157
# 쓰지 않는 zip·폴더 구분자 (data/README.md). yolo11m AI Hub 노트북과 같다.
AIHUB_EXCLUDE = ["TL_2", "TS_2"]
# 외부 데이터 폴더가 없을 때 조합 zip을 받을 팀 Drive 공유 폴더 링크(aihub_pill).
# 비워 두면 받아야 할 때 입력창으로 묻는다. AI Hub 약관상 팀원에게만 공유하는 링크라
# 넣은 채로 커밋하지 않는다.
AIHUB_URL = ""
# zip을 이미 받아 둔 폴더. 예: Colab에서 공유 폴더를 내 Drive에 바로가기로 추가한 경로
# Path("/content/drive/MyDrive/aihub_pill"). None이면 AIHUB_URL에서
# data/external/aihub_zips로 받는다.
AIHUB_ZIP_DIR = None
# 팀 Drive 공유 폴더를 내 드라이브에 바로가기로 추가한 경로(Colab). 있으면 링크로 받지 않고
# 여기서 zip을 복사한다. 링크로 받으면 Drive 다운로드 제한에 걸려 일부 파일을 못 받을 수 있다.
AIHUB_DRIVE_DIR = Path("/content/drive/MyDrive/aihub_pill") if IN_COLAB else None
# 제출에 외부 전용 클래스 예측을 넣는다. test에는 대회 56종 밖의 알약도 있어, 외부 클래스 이름
# (약 코드 숫자 = 대회 category_id 체계)을 그대로 category_id로 낸다. False면 대회 56종만 낸다.
# 학습 중 검증·최종 평가는 대회 사진이라 이 값과 관계없이 외부 클래스 예측을 지운다.
SUBMIT_EXTRA_CLASSES = True
AIHUB_TAG = f"_aihub{AIHUB_TARGET_BBOX}eda" if USE_AIHUB else ""

# 각인 OCR (5-2절). YOLO 학습이 끝난 뒤 학습 사진의 정답 bbox crop으로 따로 학습하고
# 탐지 결과의 클래스를 OCR로 다시 고른다. False면 eda 노트북과 같다.
USE_OCR = True
OCR_IMGSZ = 128  # OCR 입력 한 변(px). 정답 bbox의 긴 변을 이 크기로 맞춘다
OCR_EPOCHS = 30
OCR_BATCH = 128
OCR_LR = 0.001  # AdamW 최대 학습률(one-cycle 코사인)
OCR_PATIENCE = 10  # 검증 각인 분류 정확도가 이 에폭 동안 오르지 않으면 중단
OCR_ROTATIONS = 8  # 추론 때 crop을 360/이 값 도씩 돌려 가장 잘 읽히는 방향을 쓴다
OCR_TEMPERATURE = 1.0  # 클래스별 CTC 로그 확률을 softmax하기 전에 나누는 값
OCR_SWITCH_PROB = 0.9  # OCR 최고 클래스 확률이 이 값 이상이고
OCR_KEEP_PROB = 0.05  # YOLO가 고른 클래스의 OCR 확률이 이 값 이하일 때만 클래스를 바꾼다
TRAIN_OCR_EPOCHS = SMOKE_EPOCHS if SMOKE else OCR_EPOCHS

# 결과 폴더·제출 파일 이름에 넣을 설정 태그.
# 예: yolo26m_nmsfree_MuSGD_lr0.001_e100_p10_b16_img640_augon_mosaic1_cp0.5x2_det4_aihub157eda_ocr
RUN_TAG = (
    f"{Path(MODEL_NAME).stem}_{'nms' if NMS_ENABLED else 'nmsfree'}_{OPTIMIZER}_lr{LR0:g}"
    f"_e{TRAIN_EPOCHS}_p{PATIENCE}_b{BATCH}_img{IMGSZ}_aug{AUG}_mosaic{MOSAIC:g}"
    f"_cp{COPY_PASTE_PROB:g}x{COPY_PASTE_MAX}_det{MAX_DET}{AIHUB_TAG}{'_ocr' if USE_OCR else ''}"
)
# 실험 폴더를 모으는 곳. Colab은 Drive(0절의 COLAB_OUTPUT_ROOT), 로컬은 프로젝트 runs/.
OUTPUT_ROOT = COLAB_OUTPUT_ROOT if IN_COLAB else RUNS_DIR
# 기존 실험 폴더(이름만 적으면 OUTPUT_ROOT 아래에서 찾음). 학습이 끝난 실험이면 학습을 건너뛰고
# best.pt로 평가·제출 csv만 만들고, 중간에 끊긴 실험이면 train/weights/last.pt에서 이어 학습한다.
# 준비 파일(prepared)이 없으면(Colab 새 런타임) 같은 설정으로 다시 만들고 이전과 같은지 검사한다.
# 예: REUSE_DIR = "yolo26m_nmsfree_..._combo_holdout_s42_20261102_103918_980117"
REUSE_DIR = None
run_stamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f")
if REUSE_DIR:
    EXPERIMENT_DIR = Path(REUSE_DIR).expanduser()
    if not EXPERIMENT_DIR.is_absolute():
        EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_DIR
else:
    EXPERIMENT_DIR = (
        OUTPUT_ROOT / f"{'smoke_' if SMOKE else ''}{RUN_TAG}_combo_holdout_s{SEED}_{run_stamp}"
    )
PREPARED_DIR = (
    COLAB_PREPARED_ROOT / EXPERIMENT_DIR.name if IN_COLAB else EXPERIMENT_DIR / "prepared"
)
# Colab은 prepared가 실험 폴더 밖이라 데이터 준비가 실험 폴더를 만들지 않으므로 여기서 만든다.
if not REUSE_DIR:
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED_PATH = EXPERIMENT_DIR / "pretrained" / MODEL_NAME
if USE_AIHUB:
    for eda_csv in (AIHUB_SELECTION_CSV, AIHUB_REVIEW_CSV):
        if not eda_csv.is_file():
            raise FileNotFoundError(f"02_eda 선별·검수 csv가 없습니다: {eda_csv}")

EXPERIMENT_CONFIG = {
    "model": MODEL_NAME,
    "label": LABEL_VERSION,
    "split": SPLIT_NAME,
    "split_strategy": "single_holdout",
    "cross_validation": False,
    "n_splits": 1,
    "validation_fraction_free_combos": VAL_FRACTION,
    "seed": SEED,
    "run_tag": RUN_TAG,
    "epochs": TRAIN_EPOCHS,
    "fraction": TRAIN_FRACTION,
    "smoke": SMOKE,
    "device_type": DEVICE_TYPE,
    "patience": PATIENCE,
    "optimizer": OPTIMIZER,
    "lr0": LR0,
    "batch": BATCH,
    "imgsz": IMGSZ,
    "transfer": "full fine tuning",
    "freeze": FREEZE,
    "augmentation": AUGMENTATION,
    "mosaic": MOSAIC,
    "bbox_copy_paste": COPY_PASTE_PROB,
    "bbox_copy_paste_max": COPY_PASTE_MAX,
    "bbox_copy_paste_source": "train_gt_bbox_without_overlap",
    "bbox_copy_paste_sampling": "class_balanced_all_classes",
    "bbox_copy_paste_close_mosaic": True,
    "main_metric": MAIN_METRIC,
    "eval_conf": EVAL_CONF,
    "cm_confs": list(CM_CONFS),
    "vis_conf": VIS_CONF,
    "cm_match_iou": CM_MATCH_IOU,
    "nms": NMS_ENABLED,
    "nms_iou": NMS_IOU,
    "agnostic_nms": False,
    "max_det": MAX_DET,
    "rare_max": RARE_MAX,
    "split_unit": "combo",
    "fix_rare_train": True,
    "workers": TRAIN_WORKERS,
    "verify_validation_weights": VERIFY_VALIDATION_WEIGHTS,
    "evaluation_precision": "FP32",
    "benchmark_batch": 1,
    "benchmark_warmup": BENCHMARK_WARMUP,
    "benchmark_rounds": BENCHMARK_ROUNDS,
    "aihub": USE_AIHUB,
    "aihub_train_only": True,
    "aihub_target_bbox": AIHUB_TARGET_BBOX,
    "aihub_exclude": list(AIHUB_EXCLUDE),
    "aihub_selection": "data/processed/02_eda_selection.csv (02_eda §7·10·11)",
    "aihub_selection_sha256": file_sha256(AIHUB_SELECTION_CSV) if USE_AIHUB else None,
    "aihub_review_sha256": file_sha256(AIHUB_REVIEW_CSV) if USE_AIHUB else None,
    "aihub_whole_image": True,
    "aihub_extra_pills": "new_class",
    "aihub_problem_images": "exclude",
    "aihub_opencv_review": "human_review_applied",
    "aihub_competition_combo": "exclude",
    "ocr": USE_OCR,
    "ocr_model": "CRNN (VGG CNN + 2-layer BiLSTM) + CTC, from scratch",
    "ocr_target": "visible face (drug_dir) print_front/print_back",
    "ocr_tokens": {"십자분할선": "+", "분할선": "|", "마크": "@"},
    "ocr_imgsz": OCR_IMGSZ,
    "ocr_epochs": TRAIN_OCR_EPOCHS,
    "ocr_batch": OCR_BATCH,
    "ocr_lr": OCR_LR,
    "ocr_patience": OCR_PATIENCE,
    "ocr_rotations": OCR_ROTATIONS,
    "ocr_temperature": OCR_TEMPERATURE,
    "ocr_switch_prob": OCR_SWITCH_PROB,
    "ocr_keep_prob": OCR_KEEP_PROB,
}
# 학습 결과에 영향이 없는 기록·평가용 항목. 재사용할 때 이 항목은 달라도 된다.
# conf·NMS·max_det는 학습 중 검증(best.pt 선택)에 쓰였으므로 같아야 한다.
NON_TRAINING_KEYS = {
    "run_tag",
    "smoke",
    "device_type",
    "workers",
    "verify_validation_weights",
    "evaluation_precision",
    "benchmark_batch",
    "benchmark_warmup",
    "benchmark_rounds",
    "cm_confs",
    "vis_conf",
    "cm_match_iou",
    # OCR은 YOLO 학습 뒤 따로 학습한다. OCR 학습 설정은 5-2절이 ocr/completed.json과 따로 비교한다.
    *(key for key in EXPERIMENT_CONFIG if key.startswith("ocr")),
}


def training_config_diff(saved: dict) -> list[str]:
    """저장된 실험 설정과 현재 설정에서 학습에 영향을 주는 항목 중 값이 다른 것을 반환한다."""
    current = json.loads(json.dumps(EXPERIMENT_CONFIG))
    return sorted(
        key
        for key in saved.keys() & current.keys()
        if key not in NON_TRAINING_KEYS and saved[key] != current[key]
    )


if REUSE_DIR:
    saved_path = EXPERIMENT_DIR / "experiment.json"
    if not saved_path.is_file():
        raise FileNotFoundError(f"재사용할 실험 기록이 없습니다: {saved_path}")
    saved_config = json.loads(saved_path.read_text(encoding="utf-8"))["config"]
    if diff := training_config_diff(saved_config):
        raise ValueError(
            "재사용할 실험과 학습 설정이 다릅니다. 설정 셀을 아래 값에 맞추세요: "
            + ", ".join(f"{key}={saved_config[key]!r}" for key in diff)
        )
    print(f"기존 실험 재사용: {EXPERIMENT_DIR.name} (끝난 학습은 평가만, 끊긴 학습은 이어서)")
display(pd.Series(EXPERIMENT_CONFIG, name="실험 설정").to_frame())

In [ ]:
# CUDA·MPS가 모두 없으면 CPU로 자동 대체하지 않고 중단한다.
if get_device() != DEVICE_TYPE:
    raise RuntimeError("설정 셀 이후 장치가 바뀌었습니다. 설정 셀부터 다시 실행하세요.")
if DEVICE_TYPE not in {"cuda", "mps"}:
    raise RuntimeError(
        "CUDA GPU나 Apple MPS를 사용할 수 없습니다. 프로젝트 uv 커널과 "
        "CUDA 지원 PyTorch(NVIDIA) 또는 MPS 지원 PyTorch(Mac)를 확인하세요."
    )
if DEVICE_TYPE == "cuda" and not 0 <= GPU_INDEX < torch.cuda.device_count():
    raise ValueError(f"GPU_INDEX={GPU_INDEX}, GPU 수={torch.cuda.device_count()}")
if not {"nms", "quantize"} <= DEFAULT_CFG_DICT.keys():
    raise RuntimeError("프로젝트의 Ultralytics 8.4.182 이상 환경을 선택하세요.")
if LABEL_VERSION != "v1" or VIS_CONF not in CM_CONFS:
    raise ValueError("이 베이스라인은 v1과 혼동행렬에 포함된 표시 신뢰도를 사용합니다.")
if not 0 <= COPY_PASTE_PROB <= 1 or COPY_PASTE_MAX < 1:
    raise ValueError("COPY_PASTE_PROB는 0~1, COPY_PASTE_MAX는 1 이상이어야 합니다.")
if OCR_IMGSZ % 16 or min(OCR_EPOCHS, OCR_BATCH, OCR_PATIENCE, OCR_ROTATIONS) < 1:
    raise ValueError("OCR_IMGSZ는 16의 배수, OCR 에폭·배치·patience·회전 수는 1 이상이어야 합니다.")
if not 0 <= OCR_KEEP_PROB < OCR_SWITCH_PROB <= 1 or OCR_TEMPERATURE <= 0:
    raise ValueError("OCR 확률 기준은 0 ≤ KEEP < SWITCH ≤ 1, 온도는 0보다 커야 합니다.")
if AIHUB_TARGET_BBOX < 1:
    raise ValueError("AIHUB_TARGET_BBOX는 1 이상이어야 합니다.")
if not 0 < VAL_FRACTION < 1 or PIPELINE_PARTITION_ID != 0:
    raise ValueError("단일 holdout 비율과 공용 파이프라인 식별자를 확인하세요.")
if DEVICE_TYPE == "cuda":
    torch.cuda.set_device(GPU_INDEX)
    DEVICE = f"cuda:{GPU_INDEX}"
    gpu = torch.cuda.get_device_properties(GPU_INDEX)
    gpu_name, gpu_memory = gpu.name, gpu.total_memory
else:
    DEVICE = "mps"  # ultralytics는 mps:0이 아니라 mps를 받는다.
    gpu_name = f"Apple MPS ({platform.machine()})"
    gpu_memory = torch.mps.recommended_max_memory()
init_seeds(SEED, deterministic=True)


def sync_device() -> None:
    """속도 측정 전후로 장치의 비동기 연산이 끝날 때까지 기다린다."""
    if DEVICE_TYPE == "cuda":
        torch.cuda.synchronize(DEVICE)
    else:
        torch.mps.synchronize()


def empty_device_cache() -> None:
    """모델을 지운 뒤 장치 메모리 캐시를 비운다."""
    if DEVICE_TYPE == "cuda":
        torch.cuda.empty_cache()
    else:
        torch.mps.empty_cache()


ENVIRONMENT = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda": torch.version.cuda,
    "device": DEVICE,
    "gpu": gpu_name,
    "gpu_memory_gib": round(gpu_memory / 1024**3, 2),
    "smoke": SMOKE,
}
raw_root = PROJECT_ROOT / "data" / "raw"
RAW_DIR = raw_root / "sprint_ai_project1_data"
if not RAW_DIR.is_dir():
    RAW_DIR = raw_root
TRAIN_IMAGES = list_images(RAW_DIR / "train_images")
TEST_IMAGES = list_images(RAW_DIR / "test_images")
if (len(TRAIN_IMAGES), len(TEST_IMAGES)) != (232, 842):
    raise ValueError(
        f"확인한 원본 수와 다릅니다: train={len(TRAIN_IMAGES)}, test={len(TEST_IMAGES)}"
    )
if len({path.name for path in TRAIN_IMAGES}) != len(TRAIN_IMAGES):
    raise ValueError("중복된 train 이미지 파일명이 있습니다.")
display(pd.Series(ENVIRONMENT, name="장치 환경").to_frame())
print(f"결과 폴더: {EXPERIMENT_DIR}")

## 2. EDA 보정 라벨과 희귀 조합 train 고정

현재 v1 파이프라인은 누락 8개 추가, kwon 좌표 수정, 검수된 19232 캡슐 보정을 사용합니다.
EDA의 과거 미결정 설명과 현재 파이프라인 적용 상태는 다릅니다. 현재 v1을 따르며
여백 보정(v2)이나 새 검수 승인은 하지 않습니다.

분할 단위는 파일 이름 첫 밑줄 앞의 **알약 조합(combo)**입니다. 같은 조합의 모든 각도 사진은
train 또는 validation 한쪽에만 둡니다. 촬영 세팅이 비슷해도 조합이 다르면 따로 분할합니다.

보정 후 전체 bbox가 5개 이하인 클래스가 든 조합 전체를 train에 고정합니다.
나머지 조합은 GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)로
**한 번만 분할**합니다. 분할 후보나 교차검증 fold를 생성하지 않습니다.
8:2는 고정분을 제외한 조합 수 기준이며 조합별 이미지 수가 달라 실제 이미지 비율은 다릅니다.

라벨·YOLO 변환·목록 파일 생성은 기존 data_main.run()을 사용하고, 그 호출 동안만
분할 함수를 아래 단일 holdout 함수로 연결합니다. 실제 방식은 meta.json과 experiment.json에
기록합니다. 공용 파이프라인 파일 형식상 fold_0.csv와 fold_0/data.yaml이라는 이름을 쓰지만,
이는 하나의 분할을 저장하는 식별자일 뿐 K-fold 교차검증을 의미하지 않습니다.

In [ ]:
def write_json(path: Path, content: dict | list) -> None:
    """실험 기록을 UTF-8 JSON으로 저장한다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")


def audit_holdout(labels: pd.DataFrame, table: pd.DataFrame) -> dict:
    """단일 분할의 이미지 보존·조합 누수·희귀 조합 train 고정을 검사한다."""
    required = set(FOLD_COLS)
    if table.empty or not required <= set(table.columns):
        raise AssertionError("단일 holdout 분할 표가 없거나 필수 열이 빠졌습니다.")
    if set(table["fold"]) != {PIPELINE_PARTITION_ID}:
        raise AssertionError("공용 파이프라인에는 단일 분할 표 하나만 전달해야 합니다.")
    if table["file_name"].duplicated().any():
        raise AssertionError("분할 표에 중복 이미지가 있습니다.")
    if set(table["file_name"]) != set(labels["file_name"]):
        raise AssertionError("분할 표와 라벨의 이미지가 다릅니다.")
    if set(table["split"]) != {"train", "val"}:
        raise AssertionError("train/validation이 모두 필요합니다.")
    expected_combo = table["file_name"].str.split("_").str[0]
    if not table["combo"].eq(expected_combo).all() or not table["group"].eq(table["combo"]).all():
        raise AssertionError("이미지를 combo 단위로 묶어야 합니다.")
    if table.groupby("combo")["split"].nunique().gt(1).any():
        raise AssertionError("같은 combo가 train/validation 양쪽에 있습니다.")
    rare_ids = set(rare_classes(labels))
    rare_files = labels.loc[labels["category_id"].isin(rare_ids), "file_name"]
    rare_combos = set(rare_files.str.split("_").str[0])
    should_fix = table["combo"].isin(rare_combos)
    if not table["fixed"].eq(should_fix).all():
        raise AssertionError("희귀 알약이 포함된 조합 전체를 고정해야 합니다.")
    if not table.loc[should_fix, "split"].eq("train").all():
        raise AssertionError("희귀 조합이 validation에 포함되었습니다.")
    train_files = set(table.loc[table["split"] == "train", "file_name"])
    val_files = set(table.loc[table["split"] == "val", "file_name"])
    train_ids = set(labels.loc[labels["file_name"].isin(train_files), "category_id"])
    val_ids = set(labels.loc[labels["file_name"].isin(val_files), "category_id"])
    if train_files & val_files or rare_ids & val_ids:
        raise AssertionError("이미지가 중복되거나 희귀 클래스가 validation에 있습니다.")
    free = table.loc[~should_fix]
    return {
        "train_images": len(train_files),
        "val_images": len(val_files),
        "train_combos": int(table.loc[table["split"] == "train", "combo"].nunique()),
        "val_combos": int(table.loc[table["split"] == "val", "combo"].nunique()),
        "free_combos": int(free["combo"].nunique()),
        "fixed_train_images": int(should_fix.sum()),
        "fixed_train_groups": len(rare_combos),
        "val_image_ratio": len(val_files) / len(table),
        "free_val_combo_ratio": free.loc[free["split"] == "val", "combo"].nunique()
        / free["combo"].nunique(),
        "val_only_classes": sorted(int(value) for value in val_ids - train_ids),
        "n_classes_not_in_val": len(set(labels["category_id"]) - val_ids),
        "combo_leak": 0.0,
        "rare_in_val": 0,
    }


def build_combo_holdout(
    labels: pd.DataFrame,
    spec: SplitSpec,
    labels_sha256: str,
    val_fraction: float = VAL_FRACTION,
) -> tuple[pd.DataFrame, dict]:
    """희귀 조합을 train에 고정하고 남은 조합을 한 번만 약 8:2로 나눈다."""
    if spec.unit != "combo" or not 0 < val_fraction < 1:
        raise ValueError("combo 단위와 0~1 사이 validation 비율이 필요합니다.")
    units = pd.DataFrame({"file_name": sorted(labels["file_name"].unique())})
    units["combo"] = units["file_name"].str.split("_").str[0]
    units["group"] = units["combo"]
    rare_ids = rare_classes(labels)
    rare_files = labels.loc[labels["category_id"].isin(rare_ids), "file_name"]
    rare_combos = set(rare_files.str.split("_").str[0])
    units["fixed"] = units["combo"].isin(rare_combos)
    free = units.loc[~units["fixed"]].reset_index(drop=True)
    if free["combo"].nunique() < 2:
        raise ValueError("희귀 조합을 제외한 조합이 2개 이상이어야 단일 분할할 수 있습니다.")
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=spec.seed)
    _, val_indices = next(splitter.split(free, groups=free["combo"]))
    val_files = set(free.iloc[val_indices]["file_name"])
    table = units.assign(
        fold=PIPELINE_PARTITION_ID,
        split=units["file_name"].isin(val_files).map({True: "val", False: "train"}),
    )[FOLD_COLS]
    check = audit_holdout(labels, table)
    meta = {
        **asdict(spec),
        "fix_rare": True,
        "strategy": "single_holdout",
        "cross_validation": False,
        "n_splits": 1,
        "validation_fraction_free_combos": val_fraction,
        "rare_max": RARE_MAX,
        "rare_classes": rare_ids,
        "labels_sha256": labels_sha256,
        "n_images": len(units),
        "fixed_train_images": check["fixed_train_images"],
        "fixed_train_groups": check["fixed_train_groups"],
        "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        # step3_split()의 기록 형식에만 필요한 단일 파티션 식별자.
        "checks": [{"fold": PIPELINE_PARTITION_ID, **check}],
    }
    return table, meta


def prepare_data() -> Path:
    """기존 파이프라인으로 단일 holdout의 학습 입력 하나만 준비한다."""
    import matplotlib

    previous_backend = matplotlib.get_backend()
    try:
        from data_process import data_main
    finally:
        plt.switch_backend(previous_backend)
        # data_main은 import할 때 Agg로 바꾸고, 이때 IPython의 그림 PNG 변환기가 해제된다.
        # 백엔드만 되돌리면 plt.show()가 <Figure ...> 글자만 내므로 inline 표시를 다시 켠다.
        shell = get_ipython()
        if shell is not None and "inline" in previous_backend:
            shell.run_line_magic("matplotlib", "inline")
    if PREPARED_DIR.exists():
        raise FileExistsError(
            "이미 준비한 실험입니다. 설정 셀에서 새 실험을 만들거나 다음 셀로 진행하세요."
        )
    source_dir = PROJECT_ROOT / "data" / "processed"
    required = ("added_annotations.csv", "fixed_annotations.csv")
    for name in required:
        if not (source_dir / name).is_file():
            raise FileNotFoundError(f"기존 보정 기록이 없습니다: {source_dir / name}")
    PREPARED_DIR.mkdir(parents=True)
    for name in (*required, "padding_review.csv"):
        if (source_dir / name).is_file():
            shutil.copy2(source_dir / name, PREPARED_DIR / name)
    # 기존 소스는 수정하지 않고, run()의 출력 경로와 단일 분할 함수를 호출 동안 연결한다.
    with (
        patch.object(data_main, "PROCESSED_DIR", PREPARED_DIR),
        patch.object(data_yaml, "YOLO_ROOT", PREPARED_DIR / "yolo"),
        patch.object(data_main, "build_split", build_combo_holdout),
    ):
        paths = data_main.run(
            LABEL_VERSION, [SPLIT_NAME], use_code_names=True, n_check=0, seed=SEED
        )
        expected = data_yaml.data_yaml_path(LABEL_VERSION, SPLIT_NAME, PIPELINE_PARTITION_ID)
        if paths != [expected]:
            raise AssertionError(f"단일 holdout data.yaml 하나가 필요합니다: {paths}")
    return expected

In [ ]:
# 실행 시 데이터 준비 파일만 생성한다. 학습은 4절의 학습 셀에서 시작한다.
# REUSE_DIR이면 그 실험에서 만든 준비 파일을 그대로 쓴다. 없으면(Colab 새 런타임) 다시 만든다.
if REUSE_DIR:
    with patch.object(data_yaml, "YOLO_ROOT", PREPARED_DIR / "yolo"):
        DATA_YAML = data_yaml.data_yaml_path(LABEL_VERSION, SPLIT_NAME, PIPELINE_PARTITION_ID)
    if not DATA_YAML.is_file():
        print(f"준비 파일이 없어 같은 설정으로 다시 만듭니다: {PREPARED_DIR}")
        DATA_YAML = prepare_data()
else:
    DATA_YAML = prepare_data()
LABELS = pd.read_csv(PREPARED_DIR / "labels" / f"labels_{LABEL_VERSION}.csv")
CLASS_MAP = pd.read_csv(PREPARED_DIR / "yolo" / LABEL_VERSION / "class_map.csv")
SPLIT_TABLE = pd.read_csv(
    PREPARED_DIR / "splits" / SPLIT_NAME / f"fold_{PIPELINE_PARTITION_ID}.csv"
)
SPLIT_META = json.loads(
    (PREPARED_DIR / "splits" / SPLIT_NAME / "meta.json").read_text(encoding="utf-8")
)
if set(LABELS["file_name"]) != {path.name for path in TRAIN_IMAGES}:
    raise AssertionError("v1 라벨과 원본 train 이미지가 다릅니다.")
SPLIT_AUDIT = audit_holdout(LABELS, SPLIT_TABLE)
if REUSE_DIR:
    # 다시 만든 라벨·분할이 처음 실험과 같아야 이어 학습·평가 결과를 믿을 수 있다.
    saved_split = json.loads((EXPERIMENT_DIR / "experiment.json").read_text(encoding="utf-8"))[
        "split"
    ]
    for key in ("labels_sha256", "checks"):
        if json.loads(json.dumps(SPLIT_META[key])) != saved_split[key]:
            raise AssertionError(f"재사용 실험과 라벨·분할이 다릅니다({key}). 데이터를 확인하세요.")
display(pd.Series(SPLIT_AUDIT, name="단일 holdout 검사").to_frame())
print("교차검증 없음: combo 단위의 train/validation 단일 분할")
print(f"적용 라벨: {LABEL_VERSION}, bbox={len(LABELS)}, 클래스={len(CLASS_MAP)}")
print(f"희귀 클래스(bbox {RARE_MAX}개 이하): {rare_classes(LABELS)}")
print(
    f"희귀 조합 train 고정: {SPLIT_META['fixed_train_images']}장 / "
    f"{SPLIT_META['fixed_train_groups']}조합"
)
print(
    f"실제 이미지당 최대 정답 수: {LABELS.groupby('file_name').size().max()} / 탐지 제한: {MAX_DET}"
)
print("train 전용 희귀 클래스의 validation AP는 평가 불가(NaN)로 기록합니다.")
pd.DataFrame([SPLIT_AUDIT]).to_csv(EXPERIMENT_DIR / "holdout_audit.csv", index=False)
if not REUSE_DIR:  # 재사용할 때는 원래 실험 기록을 덮어쓰지 않는다.
    write_json(
        EXPERIMENT_DIR / "experiment.json",
        {
            "config": EXPERIMENT_CONFIG,
            "environment": ENVIRONMENT,
            "split": SPLIT_META,
            "label_corrections": list(LABEL_VERSIONS[LABEL_VERSION]),
            "source_correction_sha256": {
                path.name: file_sha256(path) for path in PREPARED_DIR.glob("*.csv")
            },
        },
    )

## 2-2. AI Hub 외부 데이터 추가 (02_eda 선별·검수 csv)

`02_eda.ipynb`(exp/aihub_eda 브랜치)가 고르고 사람이 검수한 외부 bbox 목록을 읽어, 그 bbox가 든 사진을
**train에만** 더합니다. validation은 위에서 만든 대회 사진 그대로입니다.

1. **선별 (02_eda §5~7)**: `data/processed/02_eda_selection.csv`의 bbox를 그대로 씁니다. 02_eda가 대회 v1과
   공통인 클래스마다 157개가 되도록 seed 42로 뽑고 §10~11 검수를 마친 목록입니다(좌표는 승인 보정 반영).
   csv의 bbox가 외부 JSON에 있는지, 좌표가 원본 JSON + 승인 보정과 같은지, 목표를 넘는 클래스가 없는지 검사합니다.
2. **검수 결과 적용 (02_eda §10~11)**: `data/processed/02_eda_review_decisions.csv`의 `승인`은 `new_bbox`로
   고치고(원본 좌표가 `old_bbox`와 같은지 확인), `원본 유지`는 그대로 둡니다. `이미지 제외`(bbox 겹침·라벨 누락·
   잘림 및 SAM 재생성 실패) 사진은 선별 csv에 없어야 합니다.
3. **안전 검사 (02_eda §8)**: 승인 보정을 반영한 사진 전체 라벨로 문제 사진 검사(라벨 누락·파일명에 없는 라벨·
   bbox 겹침·좌표 오류·JSON과 실제 크기 다름)와 대회와 같은 조합 검사를 다시 하고, 걸린 사진은 뺍니다.
   제외 수는 아래 표에 나옵니다.
4. **라벨**: 사진을 통째로 쓰므로 함께 찍힌 알약도 모두 라벨링합니다(선별하지 않은 bbox는 원본 JSON 좌표).
   대회 56종에 없는 외부 전용 알약은 대회 번호(0~55) 뒤에 이어지는 **새 클래스**로 학습합니다.
5. **YOLO 변환**: 외부 사진 링크와 txt는 `prepared/yolo/v1_aihub/`에 따로 만들고 대회 파이프라인
   폴더(`prepared/yolo/v1`)는 건드리지 않습니다. `train.txt` = 대회 train + 외부 사진,
   `val.txt`는 대회 파이프라인 파일 그대로, `nc`·`names`는 외부 전용 클래스만큼 늘립니다.

외부 데이터는 푼 폴더(`.env`의 `AIHUB_EXTRACT_DIR`, 없으면 `data/external/aihub_combo`)에서 읽습니다.
선별 csv에 VS_1(검증용) 사진도 있으므로 VS·VL zip도 풀어 둬야 합니다.
그 폴더가 없거나 비어 있으면 팀 Drive 공유 폴더에서 조합 zip을 `data/external/aihub_zips/`로 받아
`AIHUB_EXCLUDE`를 뺀 zip만 풉니다. 링크는 설정 셀의 `AIHUB_URL`, 비어 있으면 입력창으로 받습니다.
zip을 이미 받아 둔 폴더가 있으면 설정 셀의 `AIHUB_ZIP_DIR`에 넣어 다운로드를 건너뜁니다.
Colab에서 공유 폴더 바로가기(`AIHUB_DRIVE_DIR`)가 있으면 링크 대신 거기서 빠진 zip만 복사합니다.
링크는 AI Hub 약관상 팀원에게만 공유하므로(data/README.md) 넣은 채로 커밋하지 않습니다.
5-2절 각인 OCR을 위해 외부 라벨 csv에 보이는 면·각인 열(`drug_dir`, `print_front`, `print_back`)도 남깁니다.
쓴 bbox·라벨·제외 사유 표는 `prepared/yolo/v1_aihub/`의 csv와 `aihub_info.json`,
`experiment.json`의 `aihub`에 남습니다.

In [ ]:
AIHUB_BOX_NUMBERS = ["x", "y", "w", "h", "width", "height"]
# 5-2절 각인 OCR 정답(보이는 면과 앞·뒷면 각인). 외부 라벨 csv에도 남겨 재사용할 때 쓴다.
AIHUB_TEXT_COLUMNS = ["drug_dir", "print_front", "print_back"]
AIHUB_YOLO_DIR = PREPARED_DIR / "yolo" / f"{LABEL_VERSION}_aihub"
TRAIN_DATA_YAML_PATH = (
    AIHUB_YOLO_DIR / "splits" / SPLIT_NAME / f"fold_{PIPELINE_PARTITION_ID}" / "data.yaml"
)
N_COMP_CLASSES = len(CLASS_MAP)  # yolo_idx가 이 값 이상이면 외부 전용 클래스


def combo_of(file_name: str) -> str:
    """파일명 첫 밑줄 앞의 알약 조합(K-001900-003544-...)."""
    return file_name.split("_")[0]


def combo_codes(file_name: str) -> set[int]:
    """파일명 조합의 약 코드 집합. 사진에서 센 알약 수가 아니다."""
    return {int(code) for code in combo_of(file_name).split("-")[1:]}


def invalid_boxes(boxes: pd.DataFrame) -> pd.Series:
    """좌표 결측·크기 0 이하·이미지 경계 초과·같은 약 중복 라벨인 bbox (02_eda §6)."""
    numeric = boxes[AIHUB_BOX_NUMBERS].apply(pd.to_numeric, errors="coerce")
    flags = bbox_flags(boxes)
    return (
        ~np.isfinite(numeric).all(axis=1)
        | numeric[["width", "height"]].le(0).any(axis=1)
        | flags["out_of_image"]
        | flags["non_positive"]
        | boxes.duplicated(["file_name", "category_id"], keep=False)
    )


def exclusion_reasons(boxes: pd.DataFrame, paths: pd.Series, comp_combos: set[str]) -> pd.DataFrame:
    """외부 사진마다 train에서 뺄 사유를 표로 만든다 (02_eda §8, yolo11m AI Hub 노트북과 같음)."""
    files = sorted(boxes["file_name"].unique())
    boxes = boxes.astype({"category_id": int})
    numeric = boxes[AIHUB_BOX_NUMBERS].apply(pd.to_numeric, errors="coerce")
    positive = np.isfinite(numeric).all(axis=1) & numeric[["w", "h", "width", "height"]].gt(0).all(
        axis=1
    )
    pairs = find_box_pairs(boxes[positive], max_gap=0)
    labeled = boxes.groupby("file_name")["category_id"].agg(set)
    # 헤더 표의 file_name은 디스크 이름(NFD일 수 있음)이라 순서로 맞춘다.
    headers = read_image_headers([Path(paths[name]) for name in files], "aihub")
    real_size = headers[["width", "height"]].to_numpy(dtype=float)
    json_size = boxes.groupby("file_name")[["width", "height"]].first().loc[files]
    reasons = pd.DataFrame(index=pd.Index(files, name="file_name"))
    reasons["라벨 누락"] = [bool(combo_codes(name) - labeled[name]) for name in files]
    reasons["파일명에 없는 라벨"] = [bool(labeled[name] - combo_codes(name)) for name in files]
    reasons["bbox 겹침"] = reasons.index.isin(set(pairs.loc[pairs["iou"].gt(0), "file_name"]))
    reasons["좌표·중복 오류"] = reasons.index.isin(
        set(boxes.loc[invalid_boxes(boxes), "file_name"])
    )
    reasons["JSON·실제 크기 다름"] = ~(real_size == json_size.to_numpy(dtype=float)).all(axis=1)
    # 대회 사진과 약 조합이 같으면 같은 조합의 대회 validation 사진과 겹칠 수 있다
    reasons["대회와 같은 조합"] = [combo_of(name) in comp_combos for name in files]
    return reasons


AIHUB_KEY = ["source_json", "annotation_id"]  # 외부 bbox 식별자 (load_annotations)
REVIEW_ACTIONS = {"승인", "원본 유지", "이미지 제외"}


def with_key(table: pd.DataFrame) -> pd.DataFrame:
    """파일명·JSON 경로를 NFC로, annotation_id를 정수로 맞춰 csv와 외부 라벨을 이을 수 있게 한다."""
    table = table.copy()
    for column in ["file_name", "source_json"]:
        table[column] = [
            unicodedata.normalize("NFC", value) if isinstance(value, str) else value
            for value in table[column]
        ]
    table["annotation_id"] = pd.to_numeric(table["annotation_id"], errors="coerce").astype("Int64")
    return table


def read_eda_csv(path: Path) -> pd.DataFrame:
    """02_eda가 만든 팀 공유 csv(BOM 포함)를 읽는다."""
    return with_key(pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, na_values=[""]))


def parse_bbox(value: object) -> list[float]:
    """검수 csv의 '[x, y, w, h]' 문자열을 유한한 숫자 4개로 읽는다."""
    numbers = [float(number) for number in json.loads(str(value))]
    if len(numbers) != 4 or not np.isfinite(numbers).all():
        raise ValueError(f"bbox 형식이 아닙니다: {value}")
    return numbers


def apply_eda_review(boxes: pd.DataFrame, review: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """02_eda §10~11에서 승인한 bbox만 new_bbox로 고친다. 원본 좌표가 old_bbox와 달라도 멈춘다."""
    boxes = boxes.copy()
    approved = review[review["action"].eq("승인")]
    if approved["annotation_id"].isna().any() or approved.duplicated(AIHUB_KEY).any():
        raise ValueError("승인 행의 bbox 식별자가 비었거나 중복됩니다.")
    position = pd.Series(
        range(len(boxes)), index=pd.MultiIndex.from_frame(boxes[AIHUB_KEY].astype(object))
    )
    n_corrected = 0
    for row in approved.itertuples():
        index = position.get((row.source_json, int(row.annotation_id)))
        if index is None:
            continue  # 쓰지 않는 사진의 승인
        current = boxes.iloc[index]
        if current["file_name"] != row.file_name or int(current["category_id"]) != int(
            row.category_id
        ):
            raise ValueError(f"검수 csv와 외부 라벨의 사진·클래스가 다릅니다: {row.source_json}")
        if not np.allclose(current[["x", "y", "w", "h"]].astype(float), parse_bbox(row.old_bbox)):
            raise ValueError(f"원본 좌표가 검수 때(old_bbox)와 다릅니다: {row.source_json}")
        new_box = parse_bbox(row.new_bbox)
        boxes.iloc[index, [boxes.columns.get_loc(name) for name in ["x", "y", "w", "h"]]] = new_box
        boxes.iloc[index, boxes.columns.get_loc("area")] = new_box[2] * new_box[3]
        n_corrected += 1
    return boxes, n_corrected


def env_path(name: str, default: str) -> Path:
    """.env의 경로 값(없으면 default)을 프로젝트 루트 기준 절대 경로로 바꾼다."""
    path = Path(os.getenv(name) or default).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path


def extracted_folders(extract_dir: Path) -> list[Path]:
    """extract_dir 아래 다 푼 외부 데이터 폴더 중 AIHUB_EXCLUDE에 없는 것."""
    if not extract_dir.is_dir():
        return []
    return sorted(
        folder
        for folder in extract_dir.iterdir()
        if folder.is_dir()
        and not folder.name.startswith(".")
        and zip_key(folder) not in AIHUB_EXCLUDE
    )


def copy_drive_zips(drive_dir: Path, zip_dir: Path) -> None:
    """Drive 바로가기 폴더의 조합 zip 중 빠졌거나 크기가 다른 것만 zip_dir로 복사한다."""
    zip_dir.mkdir(parents=True, exist_ok=True)
    sources = sorted(
        path
        for path in drive_dir.glob("*.zip")
        if COMBO_KEYWORD in unicodedata.normalize("NFC", path.name)
        and zip_key(path) not in AIHUB_EXCLUDE
    )
    if not sources:
        raise FileNotFoundError(f"Drive 바로가기 폴더에 조합 zip이 없습니다: {drive_dir}")
    print(f"Drive 바로가기에서 zip을 복사합니다: {drive_dir} → {zip_dir}")
    for source in sources:
        target = zip_dir / unicodedata.normalize("NFC", source.name)
        if target.is_file() and target.stat().st_size == source.stat().st_size:
            continue
        print(f"  복사: {target.name} ({source.stat().st_size / 1e9:.1f}GB)")
        partial = target.with_name(f".{target.name}.partial")
        shutil.copyfile(source, partial)
        partial.rename(target)


def download_aihub(extract_dir: Path) -> None:
    """팀 Drive 공유 폴더의 조합 zip을 준비해 AIHUB_EXCLUDE를 뺀 것만 푼다.

    설정 셀의 AIHUB_ZIP_DIR이 있으면 그 폴더의 zip을 그대로 쓴다. 없으면 공유 폴더 바로가기
    (AIHUB_DRIVE_DIR)에서 빠진 zip만 복사하고, 바로가기도 없으면 AIHUB_URL(비어 있으면
    입력창)로 받는다. 이미 받은 파일·다 푼 zip은 건너뛴다(aihub_data.py).
    """
    if AIHUB_ZIP_DIR:
        zip_dir = Path(AIHUB_ZIP_DIR).expanduser()
    else:
        zip_dir = PROJECT_ROOT / "data" / "external" / "aihub_zips"
        drive_dir = Path(AIHUB_DRIVE_DIR).expanduser() if AIHUB_DRIVE_DIR else None
        if drive_dir is not None and drive_dir.is_dir():
            copy_drive_zips(drive_dir, zip_dir)
        else:
            url = AIHUB_URL or getpass(
                "AI Hub zip 폴더 Drive 링크 AIHUB_URL (화면에 표시되지 않음): "
            )
            if not url.strip():
                raise FileNotFoundError(
                    f"외부 데이터 폴더가 없고 받을 주소도 없습니다: {extract_dir}. "
                    "팀 Drive aihub_pill 폴더 공유 링크를 넣으세요."
                )
            print(f"외부 데이터가 없어 팀 Drive에서 zip을 받습니다 → {zip_dir}")
            try:
                download_drive_folder(url.strip(), zip_dir)
            except Exception as error:
                raise RuntimeError(
                    "Drive에서 일부 zip을 받지 못했습니다(큰 공유 파일의 다운로드 제한일 수 있음). "
                    "공유 폴더를 내 드라이브에 바로가기로 추가하고 설정 셀의 AIHUB_DRIVE_DIR을 "
                    "그 경로로 맞춘 뒤 설정 셀부터 다시 실행하세요. "
                    "이미 받은 zip은 다시 받지 않습니다."
                ) from error
    zips = find_combo_zips(zip_dir, exclude=AIHUB_EXCLUDE)
    if not zips:
        raise FileNotFoundError(
            f"조합 zip이 없습니다: {zip_dir}. Drive 폴더의 파일이 '링크가 있는 모든 사용자'로 "
            "공유되어 있는지 확인하세요."
        )
    extract_zips(zips, extract_dir)


def aihub_folders() -> tuple[Path, list[Path]]:
    """02_eda §1과 같은 위치에서 다 푼 외부 데이터 폴더 중 AIHUB_EXCLUDE에 없는 것을 고른다.

    폴더가 없거나 쓸 하위 폴더가 없으면 팀 Drive에서 받아 푼다(download_aihub).
    """
    load_dotenv(PROJECT_ROOT / ".env")
    extract_dir = env_path("AIHUB_EXTRACT_DIR", "data/external/aihub_combo")
    folders = extracted_folders(extract_dir)
    if not folders:
        download_aihub(extract_dir)
        folders = extracted_folders(extract_dir)
    if not folders:
        raise FileNotFoundError(f"분석할 하위 폴더가 없습니다: {extract_dir}, 제외={AIHUB_EXCLUDE}")
    return extract_dir, folders


def select_aihub() -> dict:
    """02_eda 선별·검수 csv의 bbox가 든 사진을 고르고 승인 보정을 적용해 train에 더할 라벨을 만든다.

    02_eda §8 문제 사진 검사와 대회와 같은 조합 검사는 보정 뒤 사진 전체 라벨로 다시 해서 뺀다.
    """
    extract_dir, folders = aihub_folders()
    # §1·2: 이미지와 라벨 연결. 같은 이름이 여러 폴더에 있으면 라벨을 안전하게 붙일 수 없다.
    image_index = list_combo_images(folders)
    if image_index.empty:
        raise FileNotFoundError("외부 이미지가 없습니다. TS 원천 이미지 폴더를 확인하세요.")
    if image_index["file_name"].duplicated().any():
        raise ValueError("외부 이미지 이름이 중복됩니다. 02_eda §1에서 중복 목록을 확인하세요.")
    ann_df, _ = load_combo_annotations(folders)
    if ann_df.empty or ann_df["category_id"].isna().any():
        raise ValueError("외부 라벨이 없거나 약 코드를 해석할 수 없는 bbox가 있습니다 (02_eda §2).")
    available = with_key(ann_df[ann_df["file_name"].isin(set(image_index["file_name"]))])
    paths = image_index.set_index("file_name")["path"]
    if available[AIHUB_KEY].isna().any(axis=None) or available.duplicated(AIHUB_KEY).any():
        raise ValueError(
            "외부 라벨의 bbox 식별자(source_json, annotation_id)가 비었거나 중복됩니다."
        )

    # §5: 대회 v1과 공통인 클래스마다 목표까지 부족한 bbox 수 (선별 csv 검사용)
    comp_counts = LABELS.groupby("category_id").size()
    comp_ids = {int(value) for value in comp_counts.index}
    common = sorted(comp_ids & set(available["category_id"].astype(int)))
    needed = (AIHUB_TARGET_BBOX - comp_counts).clip(lower=0)

    # §7·10·11: 02_eda가 고르고 검수한 bbox 목록과 검수 결정
    eda = read_eda_csv(AIHUB_SELECTION_CSV)
    review = read_eda_csv(AIHUB_REVIEW_CSV)
    if unknown := set(review["action"]) - REVIEW_ACTIONS:
        raise ValueError(f"알 수 없는 검수 결정이 있습니다: {sorted(unknown)}")
    if eda["annotation_id"].isna().any() or eda.duplicated(AIHUB_KEY).any():
        raise ValueError("선별 csv의 bbox 식별자가 비었거나 중복됩니다.")
    if eda.duplicated(["file_name", "category_id"]).any():
        raise ValueError("선별 csv에 같은 사진·같은 약 bbox가 두 번 있습니다.")
    review_excluded = set(review.loc[review["action"].eq("이미지 제외"), "file_name"])
    if set(eda["file_name"]) & review_excluded:
        raise ValueError("선별 csv에 검수에서 '이미지 제외'한 사진이 있습니다.")
    if not set(eda["category_id"].astype(int)) <= set(common):
        raise ValueError("선별 csv에 대회 v1과 공통이 아닌 클래스가 있습니다.")
    eda_counts = eda.groupby("category_id").size()
    if eda_counts.gt(needed.reindex(eda_counts.index).fillna(0)).any():
        raise ValueError(
            f"선별 csv에 목표 {AIHUB_TARGET_BBOX}개를 넘는 클래스가 있습니다. "
            "대회 v1 라벨이 02_eda 때와 같은지 확인하세요."
        )
    comp_names = {path.name for path in (*TRAIN_IMAGES, *TEST_IMAGES)}
    if set(eda["file_name"]) & comp_names:
        raise ValueError("선별 csv에 대회 사진과 같은 이름의 외부 사진이 있습니다.")

    # 선별 bbox가 든 사진 전체의 라벨에 승인 보정을 적용한다. 함께 찍힌 bbox도 쓴다.
    boxes = available[available["file_name"].isin(set(eda["file_name"]))].copy()
    boxes["category_id"] = boxes["category_id"].astype(int)
    boxes, n_corrected = apply_eda_review(boxes, review)
    check = eda.merge(boxes, on=AIHUB_KEY, how="left", suffixes=("_eda", ""), validate="one_to_one")
    if check["file_name"].isna().any():
        raise FileNotFoundError(
            f"선별 csv bbox {int(check['file_name'].isna().sum())}개를 외부 라벨에서 "
            "찾지 못했습니다. "
            "TL·TS·VL·VS zip을 모두 풀었는지 확인하세요."
        )
    if not (
        check["file_name_eda"].eq(check["file_name"]).all()
        and check["category_id_eda"].astype(int).eq(check["category_id"]).all()
    ):
        raise ValueError("선별 csv와 외부 라벨의 사진·클래스가 다릅니다.")
    box_columns = ["x", "y", "w", "h"]
    if not np.allclose(
        check[[f"{name}_eda" for name in box_columns]].astype(float),
        check[box_columns].astype(float),
    ):
        raise ValueError("선별 csv 좌표가 원본 JSON + 승인 보정과 다릅니다.")
    selected = boxes.merge(eda[AIHUB_KEY], on=AIHUB_KEY, validate="one_to_one")

    # §8: 보정 뒤에도 남은 문제 사진과 대회와 같은 조합의 사진은 통째로 뺀다
    comp_combos = {combo_of(name) for name in LABELS["file_name"]}
    comp_combos |= {combo_of(name) for name in comp_names if name.startswith("K-")}
    reasons = exclusion_reasons(boxes, paths, comp_combos)
    kept_files = set(reasons.index[~reasons.any(axis=1)])
    labels = boxes[boxes["file_name"].isin(kept_files)].copy()
    if set(labels.loc[labels["category_id"].isin(comp_ids), "file_name"]) != kept_files:
        raise ValueError("대회 클래스 라벨이 없는 외부 사진이 있습니다.")
    return {
        "extract_dir": extract_dir,
        "folders": folders,
        "n_images": len(image_index),
        "n_labels": len(ann_df),
        "common_classes": common,
        "comp_ids": comp_ids,
        "selected": selected,
        "review": review,
        "review_excluded": review_excluded,
        "corrected_bbox": n_corrected,
        "reasons": reasons,
        "labels": labels,
        "paths": paths,
    }


def read_list(path: Path) -> list[str]:
    """목록 txt의 이미지 경로를 빈 줄 없이 읽는다."""
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_aihub_data(selection: dict) -> tuple[Path, dict]:
    """외부 사진 링크·YOLO 라벨을 따로 만들고 train = 대회 train + 외부 사진인 data.yaml을 쓴다."""
    labels = selection["labels"]
    files = sorted(labels["file_name"].unique())
    sources = [Path(selection["paths"][name]) for name in files]
    if any(path.name != name for path, name in zip(sources, files, strict=True)):
        raise ValueError("디스크 파일 이름과 라벨 file_name이 다릅니다(NFD 등). 이름을 확인하세요.")

    # 대회 56종(yolo_idx 0~55) 뒤에 외부 전용 클래스를 약 코드 순서로 잇는다.
    extra_ids = sorted(set(labels["category_id"]) - selection["comp_ids"])
    extra_names = labels.drop_duplicates("category_id").set_index("category_id")["dl_name"]
    extra_map = pd.DataFrame(
        {
            "yolo_idx": range(N_COMP_CLASSES, N_COMP_CLASSES + len(extra_ids)),
            "category_id": extra_ids,
            "dl_name": [extra_names.get(code) for code in extra_ids],
        }
    )
    full_class_map = pd.concat([CLASS_MAP, extra_map], ignore_index=True)
    class_index = dict(
        zip(
            full_class_map["category_id"].astype(int),
            full_class_map["yolo_idx"].astype(int),
            strict=True,
        )
    )
    image_dir = AIHUB_YOLO_DIR / "images" / "train"
    label_dir = AIHUB_YOLO_DIR / "labels" / "train"
    link_counts = link_images(sources, image_dir)
    yolo_counts = write_yolo_labels(labels, files, class_index, label_dir)
    yolo_columns = ["yolo_idx", "x", "y", "w", "h"]
    max_error = 0.0
    for file_name, rows in labels.groupby("file_name"):
        txt_path = label_dir / Path(file_name).with_suffix(".txt").name
        back = read_yolo_labels(txt_path, rows["width"].iloc[0], rows["height"].iloc[0])
        expected = rows.assign(yolo_idx=rows["category_id"].map(class_index))
        if len(back) != len(expected) or expected["yolo_idx"].isna().any():
            raise ValueError(f"외부 YOLO 라벨 개수나 클래스가 다릅니다: {file_name}")
        error = np.abs(
            back[yolo_columns].sort_values(yolo_columns).to_numpy(dtype=float)
            - expected[yolo_columns].sort_values(yolo_columns).to_numpy(dtype=float)
        ).max()
        max_error = max(max_error, float(error))
    if (
        max_error > 0.01
        or yolo_counts["n_dropped"]
        or yolo_counts["n_empty"]
        or yolo_counts["n_lines"] != len(labels)
    ):
        raise ValueError(f"외부 YOLO 라벨 검사 실패: {yolo_counts}, 역변환 오차={max_error}px")

    # train = 대회 train + 외부 사진, val = 대회 파이프라인 val.txt 그대로
    base = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
    comp_train = read_list(Path(base["train"]))
    val_path = Path(base["val"])
    val_names = {Path(line).name for line in read_list(val_path)}
    comp_names = val_names | {Path(line).name for line in comp_train}
    if set(files) & comp_names or {combo_of(name) for name in files} & {
        combo_of(name) for name in val_names
    }:
        raise AssertionError("외부 사진이 대회 사진이나 validation 조합과 겹칩니다.")
    aihub_images = [str(image_dir / name) for name in files]
    out_dir = TRAIN_DATA_YAML_PATH.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    train_txt = out_dir / "train.txt"
    train_txt.write_text(
        "".join(f"{path}\n" for path in comp_train + aihub_images), encoding="utf-8"
    )
    names = {
        **{int(key): value for key, value in base["names"].items()},
        **{int(row.yolo_idx): str(row.category_id) for row in extra_map.itertuples()},
    }
    data = {
        "path": base["path"],
        "train": str(train_txt),
        "val": str(val_path),
        "nc": len(names),
        "names": names,
    }
    header = {
        "label_version": LABEL_VERSION,
        "split_name": SPLIT_NAME,
        "fold": PIPELINE_PARTITION_ID,
        "aihub_images": len(aihub_images),
        "extra_classes": len(extra_ids),
    }
    TRAIN_DATA_YAML_PATH.write_text(
        "".join(f"# {key}: {value}\n" for key, value in header.items())
        + yaml.safe_dump(data, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )

    keep_columns = ["file_name", "zip_name", "source_json", "annotation_id", "category_id"]
    keep_columns += ["dl_name", *AIHUB_BOX_NUMBERS, *AIHUB_TEXT_COLUMNS]
    selection["selected"][keep_columns].to_csv(
        AIHUB_YOLO_DIR / "aihub_selected.csv", index=False, encoding="utf-8"
    )
    labels[keep_columns].to_csv(AIHUB_YOLO_DIR / "aihub_labels.csv", index=False, encoding="utf-8")
    full_class_map.to_csv(AIHUB_YOLO_DIR / "aihub_class_map.csv", index=False, encoding="utf-8")
    reasons = selection["reasons"]
    reasons.to_csv(AIHUB_YOLO_DIR / "aihub_excluded.csv", encoding="utf-8")
    info = {
        "used": True,
        "source": "data/processed/02_eda_selection.csv·02_eda_review_decisions.csv "
        "(notebooks/02_eda.ipynb §7·10·11, exp/aihub_eda)",
        "eda_csv_sha256": {
            path.name: file_sha256(path) for path in (AIHUB_SELECTION_CSV, AIHUB_REVIEW_CSV)
        },
        "review_actions": {
            key: int(value) for key, value in selection["review"]["action"].value_counts().items()
        },
        "review_excluded_images": len(selection["review_excluded"]),
        "corrected_bbox": selection["corrected_bbox"],
        "extract_dir": str(selection["extract_dir"]),
        "folders": [folder.name for folder in selection["folders"]],
        "exclude": list(AIHUB_EXCLUDE),
        "target_bbox": AIHUB_TARGET_BBOX,
        "seed": SEED,
        "external_images": selection["n_images"],
        "external_labels": selection["n_labels"],
        "common_classes": len(selection["common_classes"]),
        "selected_bbox": len(selection["selected"]),
        "selected_images": len(reasons),
        "excluded_images": {key: int(value) for key, value in reasons.sum().items()},
        "excluded_images_total": int(reasons.any(axis=1).sum()),
        "train_images": len(aihub_images),
        "train_labels": len(labels),
        "train_labels_competition": int(labels["category_id"].isin(selection["comp_ids"]).sum()),
        "extra_classes": len(extra_ids),
        "extra_class_ids": [int(code) for code in extra_ids],
        "competition_train_images": len(comp_train),
        "val_images": len(val_names),
        "train_ratio": round((len(comp_train) + len(aihub_images)) / len(comp_train), 1),
        "link_counts": link_counts,
        "yolo_counts": yolo_counts,
        "yolo_max_roundtrip_error_px": max_error,
    }
    write_json(AIHUB_YOLO_DIR / "aihub_info.json", info)
    return TRAIN_DATA_YAML_PATH, info

In [ ]:
# 외부 데이터 준비. USE_AIHUB=False면 대회 데이터만 쓴다.
# REUSE_DIR이면 그 실험에서 만든 외부 라벨·목록을 그대로 쓴다(외부 폴더가 없어도 된다).
# 준비 파일이 없으면(Colab 새 런타임) 같은 방법으로 다시 고르고 처음 실험 기록과 같은지 검사한다.
AIHUB_SAME_KEYS = [
    "eda_csv_sha256",
    "corrected_bbox",
    "selected_bbox",
    "selected_images",
    "excluded_images",
    "train_images",
    "train_labels",
    "extra_class_ids",
    "competition_train_images",
    "val_images",
]
if not USE_AIHUB:
    TRAIN_DATA_YAML = DATA_YAML
    AIHUB_LABELS = pd.DataFrame(columns=["file_name", "category_id"])
    AIHUB_INFO = {"used": False}
    print("USE_AIHUB=False: 외부 데이터 없이 대회 데이터만 학습합니다.")
elif REUSE_DIR and TRAIN_DATA_YAML_PATH.is_file():
    TRAIN_DATA_YAML = TRAIN_DATA_YAML_PATH
    AIHUB_LABELS = pd.read_csv(AIHUB_YOLO_DIR / "aihub_labels.csv")
    AIHUB_INFO = json.loads((AIHUB_YOLO_DIR / "aihub_info.json").read_text(encoding="utf-8"))
elif REUSE_DIR:
    print(f"외부 데이터 준비 파일이 없어 같은 방법으로 다시 만듭니다: {AIHUB_YOLO_DIR}")
    saved_aihub = json.loads((EXPERIMENT_DIR / "experiment.json").read_text(encoding="utf-8"))
    saved_aihub = saved_aihub.get("aihub", {})
    AIHUB_SELECTION = select_aihub()
    TRAIN_DATA_YAML, AIHUB_INFO = write_aihub_data(AIHUB_SELECTION)
    AIHUB_LABELS = AIHUB_SELECTION["labels"]
    if diff := [key for key in AIHUB_SAME_KEYS if AIHUB_INFO.get(key) != saved_aihub.get(key)]:
        raise AssertionError(
            f"재사용 실험과 외부 데이터가 다릅니다: {diff}. 외부 zip을 확인하세요."
        )
else:
    if TRAIN_DATA_YAML_PATH.exists():
        raise FileExistsError(
            "이미 외부 데이터를 준비한 실험입니다. 새 실험은 설정 셀부터 실행하세요."
        )
    AIHUB_SELECTION = select_aihub()
    TRAIN_DATA_YAML, AIHUB_INFO = write_aihub_data(AIHUB_SELECTION)
    AIHUB_LABELS = AIHUB_SELECTION["labels"]
    experiment_path = EXPERIMENT_DIR / "experiment.json"
    experiment = json.loads(experiment_path.read_text(encoding="utf-8"))
    experiment["aihub"] = AIHUB_INFO
    write_json(experiment_path, experiment)

if USE_AIHUB:
    rows = [
        ("외부 이미지 (푼 폴더 전체)", AIHUB_INFO["external_images"]),
        ("외부 bbox 라벨 (푼 폴더 전체)", AIHUB_INFO["external_labels"]),
        ("공통 클래스 (대회 v1 ∩ 외부)", AIHUB_INFO["common_classes"]),
        ("02_eda 검수에서 뺀 사진 (선별 csv에 없음)", AIHUB_INFO["review_excluded_images"]),
        ("02_eda 선별 csv bbox", AIHUB_INFO["selected_bbox"]),
        ("  검수 승인으로 좌표를 고친 bbox", AIHUB_INFO["corrected_bbox"]),
        ("선별 bbox가 든 사진", AIHUB_INFO["selected_images"]),
        *[(f"  제외: {key}", value) for key, value in AIHUB_INFO["excluded_images"].items()],
        ("제외한 사진 (사유가 여럿이어도 한 번)", AIHUB_INFO["excluded_images_total"]),
        ("train에 더할 외부 사진", AIHUB_INFO["train_images"]),
        ("외부 학습 라벨 bbox (전체)", AIHUB_INFO["train_labels"]),
        ("  그중 대회 클래스", AIHUB_INFO["train_labels_competition"]),
        ("외부 전용 새 클래스 수", AIHUB_INFO["extra_classes"]),
        (
            "대회 train / validation 사진",
            f"{AIHUB_INFO['competition_train_images']} / {AIHUB_INFO['val_images']}",
        ),
        ("train 배수 (에폭당 시간도 대략 이만큼)", AIHUB_INFO["train_ratio"]),
    ]
    display(pd.DataFrame(rows, columns=["항목", "값"]).astype({"값": str}).set_index("항목"))
    split_of = LABELS["file_name"].map(SPLIT_TABLE.set_index("file_name")["split"])
    class_counts = CLASS_MAP.set_index("category_id")[["dl_name"]].copy()
    class_counts["대회 v1 전체"] = LABELS.groupby("category_id").size()
    class_counts["대회 train"] = LABELS[split_of.eq("train")].groupby("category_id").size()
    class_counts["외부 학습 라벨"] = AIHUB_LABELS.groupby("category_id").size()
    count_columns = ["대회 v1 전체", "대회 train", "외부 학습 라벨"]
    class_counts[count_columns] = class_counts[count_columns].fillna(0).astype(int)
    class_counts["train 합계"] = class_counts["대회 train"] + class_counts["외부 학습 라벨"]
    class_counts["대회+외부 합계"] = class_counts["대회 v1 전체"] + class_counts["외부 학습 라벨"]
    class_counts = class_counts.sort_values("대회 v1 전체")
    class_counts.to_csv(EXPERIMENT_DIR / "aihub_class_counts.csv")
    n_short = int(class_counts["대회+외부 합계"].lt(AIHUB_TARGET_BBOX).sum())
    print(f"대회+외부 합계가 {AIHUB_TARGET_BBOX}개 미만인 클래스: {n_short}/{len(class_counts)}종")
    with pd.option_context("display.max_rows", None):
        display(class_counts)
    ax = class_counts[["대회 train", "외부 학습 라벨"]].plot.barh(
        stacked=True, figsize=(11, 14), color=["steelblue", "darkorange"]
    )
    ax.axvline(AIHUB_TARGET_BBOX, color="red", linestyle="--")
    ax.legend(["competition train", "AI Hub (train only)", f"target {AIHUB_TARGET_BBOX}"])
    ax.set(title="Boxes per class in train", xlabel="boxes", ylabel="category_id")
    plt.tight_layout()
    plt.savefig(EXPERIMENT_DIR / "aihub_class_counts.png", dpi=150)
    plt.show()
    plt.close()
print(f"학습 data.yaml: {TRAIN_DATA_YAML}")
print(f"평가 validation 목록 (대회 사진만): {DATA_YAML.parent / 'val.txt'}")

In [ ]:
# 외부 학습 사진 예시. 초록 = 대회 클래스, 주황 = 외부 전용 새 클래스.
if USE_AIHUB and len(AIHUB_LABELS):
    aihub_image_dir = AIHUB_YOLO_DIR / "images" / "train"
    comp_codes = set(CLASS_MAP["category_id"])
    extra_files = sorted(
        set(AIHUB_LABELS.loc[~AIHUB_LABELS["category_id"].isin(comp_codes), "file_name"])
    )
    preview_pool = extra_files or sorted(set(AIHUB_LABELS["file_name"]))
    preview_files = pd.Series(preview_pool).sample(n=min(4, len(preview_pool)), random_state=SEED)
    fig, axes = plt.subplots(1, len(preview_files), figsize=(4 * len(preview_files), 5.5))
    for axis, file_name in zip(np.atleast_1d(axes), preview_files, strict=True):
        bgr = cv2.imread(str(aihub_image_dir / file_name))
        if bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {file_name}")
        axis.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        for box in AIHUB_LABELS[AIHUB_LABELS["file_name"] == file_name].itertuples():
            color = "lime" if box.category_id in comp_codes else "darkorange"
            axis.add_patch(
                Rectangle((box.x, box.y), box.w, box.h, fill=False, edgecolor=color, lw=2)
            )
            axis.text(box.x, box.y - 8, str(box.category_id), color=color, fontsize=8)
        axis.set_title(combo_of(file_name), fontsize=8)
        axis.axis("off")
    fig.suptitle("AI Hub train: lime = competition class, orange = external-only class")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 3. 검증 보호·주 지표·조기 중단

mAP75–95는 IoU 0.75, 0.80, 0.85, 0.90, 0.95의 클래스별 AP 평균입니다.
기존 Map7595Trainer로 best.pt와 early stopping 기준을 일치시킵니다.
0점이 이어져도 PATIENCE(10)에폭간 개선이 없으면 중단하도록 stopper를 연결합니다.

혼동행렬의 conf는 명시적으로 적용합니다. 정답 매칭 IoU=0.45는 라이브러리 기본값이며
NMS IoU=0.5와 별도로 기록합니다. Ultralytics Precision/Recall 요약은 IoU=0.5의
max-F1 신뢰도 지점 macro 평균입니다. 고정 conf에서의 혼동행렬 micro P/R은 별도 열입니다.

In [ ]:
class ExactConfusionMatrix(ConfusionMatrix):
    """요청한 신뢰도로 먼저 걸러 혼동행렬의 임계값을 명확하게 적용한다."""

    def process_batch(
        self,
        detections: dict[str, torch.Tensor],
        batch: dict,
        conf: float = 0.25,
        iou_thres: float = CM_MATCH_IOU,
    ) -> None:
        mask = detections["conf"] > conf
        filtered = {key: value[mask] for key, value in detections.items()}
        super().process_batch(filtered, batch, conf=0.0, iou_thres=iou_thres)


class ReadOnlyValidator(DetectionValidator):
    """검증에서 가중치가 바뀌지 않는지 확인하고 평가에 쓴 예측 박스를 보관한다."""

    @torch.inference_mode()
    def __call__(self, trainer=None, model=None, **kwargs) -> dict | None:
        protected = []
        if VERIFY_VALIDATION_WEIGHTS:
            candidates = (
                [trainer.model, trainer.ema.ema]
                if trainer is not None
                else [model]
                if isinstance(model, torch.nn.Module)
                else []
            )
            for candidate in candidates:
                snapshot = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in candidate.state_dict().items()
                }
                protected.append((candidate, snapshot))
        stats = super().__call__(trainer=trainer, model=model, **kwargs)
        for candidate, snapshot in protected:
            current = candidate.state_dict()
            changed = [
                name
                for name, tensor in snapshot.items()
                if name not in current or not torch.equal(tensor, current[name].detach().cpu())
            ]
            if changed or snapshot.keys() != current.keys():
                raise AssertionError(f"검증 중 가중치/버퍼가 바뀌었습니다: {changed[:5]}")
        return stats

    def init_metrics(self, model: torch.nn.Module) -> None:
        if torch.is_grad_enabled() or model.training:
            raise AssertionError("검증은 gradient 비활성화 + eval 모드여야 합니다.")
        super().init_metrics(model)
        # NMS_ENABLED=True: one2many 헤드 + 클래스별 NMS, False: one2one 헤드(end2end)
        if self.end2end == NMS_ENABLED or self.args.nms is not NMS_ENABLED:
            raise AssertionError("NMS_ENABLED와 검증 후처리 경로(NMS / NMS-free)가 다릅니다.")
        if self.args.agnostic_nms:
            raise AssertionError("클래스별 후처리가 필요합니다(agnostic_nms=False).")
        if self.args.max_det != MAX_DET or self.args.iou != NMS_IOU:
            raise AssertionError("합의한 NMS IoU/max_det와 다릅니다.")
        self.confusion_matrix = ExactConfusionMatrix(names=self.names)
        self.validation_predictions = []

    def postprocess(self, preds: torch.Tensor) -> list[dict[str, torch.Tensor]]:
        outputs = super().postprocess(preds)
        if any(len(output["conf"]) > MAX_DET for output in outputs):
            raise AssertionError("max_det 제한이 적용되지 않았습니다.")
        # 외부 전용 클래스는 대회 클래스가 아니므로 평가·시각화 예측에서 지운다.
        return [
            {key: value[output["cls"] < N_COMP_CLASSES] for key, value in output.items()}
            for output in outputs
        ]

    def update_metrics(self, preds: list[dict], batch: dict) -> None:
        super().update_metrics(preds, batch)
        if self.training:
            return
        # 별도 predict() 대신 평가한 박스를 원본 이미지 좌표로 되돌려 시각화에 사용한다.
        for index, prediction in enumerate(preds):
            prepared = self._prepare_batch(index, batch)
            original = self.scale_preds(prediction, prepared)
            self.validation_predictions.append(
                {
                    "file_name": Path(prepared["im_file"]).name,
                    "xyxy": original["bboxes"].detach().cpu().tolist(),
                    "yolo_idx": original["cls"].int().cpu().tolist(),
                    "conf": original["conf"].detach().cpu().tolist(),
                }
            )

    def finalize_metrics(self) -> None:
        super().finalize_metrics()
        self.metrics.validation_predictions = self.validation_predictions


class StrictEarlyStopping(EarlyStopping):
    """첫 평가 이후에는 엄격한 점수 증가만 개선으로 인정한다."""

    def __call__(self, epoch: int, fitness: float | None) -> bool:
        if fitness is None:
            return False
        if not np.isfinite(fitness):
            raise ValueError(f"유효하지 않은 검증 점수: {fitness}")
        if self.best_epoch == 0 or fitness > self.best_fitness:
            self.best_epoch, self.best_fitness = epoch, fitness
        delta = epoch - self.best_epoch
        self.possible_stop = delta >= self.patience - 1
        stop = delta >= self.patience
        if stop:
            print(f"Early stopping: {delta}에폭 개선 없음, best epoch={self.best_epoch}")
        return stop


# 에폭별 검증 추론시간(이미지당 ms). yolo11m 노트북과 같은 results.csv 열 이름이다.
INFERENCE_METRIC_KEY = "speed/inference_ms"


class BaselineTrainer(Map7595Trainer):
    """기존 mAP75–95 학습기에 검증 보호를 연결한다."""

    def get_validator(self) -> ReadOnlyValidator:
        return ReadOnlyValidator(
            self.test_loader,
            save_dir=self.save_dir,
            args=copy(self.args),
            _callbacks=self.callbacks,
        )

    def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
        """학습 데이터셋에만 bbox copy-paste를 붙인다. 검증 데이터셋은 그대로 둔다."""
        dataset = super().build_dataset(img_path, mode, batch)
        if mode == "train" and COPY_PASTE_PROB > 0:
            BBoxCopyPasteDataset.enable(dataset, COPY_PASTE_PROB, COPY_PASTE_MAX)
        return dataset

    def validate(self) -> tuple[dict | None, float | None]:
        metrics, fitness = super().validate()
        # 이 열 없이 시작한 결과를 이어 학습하면 results.csv 열이 어긋나므로 그때는 남기지 않는다.
        header = self.csv.read_text(encoding="utf-8").split("\n", 1)[0] if self.csv.exists() else ""
        if metrics is not None and (not header or INFERENCE_METRIC_KEY in header):
            metrics[INFERENCE_METRIC_KEY] = float(self.validator.speed["inference"])
        return metrics, fitness


class EpochTimer:
    """에폭별 학습·검증·체크포인트 저장 시간을 train/epoch_times.csv로 남긴다(yolo11m과 같은 열)."""

    def __init__(self) -> None:
        self.started_at: float | None = None
        self.active_epoch: int | None = None
        self.rows: list[dict] = []

    def on_train_start(self, trainer: BaselineTrainer) -> None:
        """이어 학습이면 이미 끝낸 에폭의 기록을 이어 쓴다."""
        path = Path(trainer.save_dir) / "epoch_times.csv"
        if trainer.start_epoch and path.is_file():
            previous = pd.read_csv(path)
            self.rows = previous[previous["epoch"] <= trainer.start_epoch].to_dict("records")

    def on_epoch_start(self, trainer: BaselineTrainer) -> None:
        """에폭을 다시 시작해도 처음 시작 시각을 유지한다."""
        if self.active_epoch != trainer.epoch + 1:
            self.active_epoch = trainer.epoch + 1
            self.started_at = perf_counter()

    def on_epoch_end(self, trainer: BaselineTrainer) -> None:
        """검증·저장까지 끝나면 기록하고, 학습 뒤 best.pt 재검증 호출은 건너뛴다."""
        if self.started_at is None:
            return
        seconds = perf_counter() - self.started_at
        self.started_at = None
        self.active_epoch = None
        elapsed = sum(row["epoch_seconds"] for row in self.rows) + seconds
        self.rows.append(
            {"epoch": trainer.epoch + 1, "epoch_seconds": seconds, "cumulative_seconds": elapsed}
        )
        pd.DataFrame(self.rows).to_csv(Path(trainer.save_dir) / "epoch_times.csv", index=False)


def start_training(trainer: BaselineTrainer) -> None:
    """엄격한 stopper를 적용하고 실제 옵티마이저·학습률 그룹을 남긴다.

    이어 학습이면 ultralytics가 last.pt에서 복원한 최고 점수·에폭을 넘겨받아 patience를 이어서 센다.
    """
    restored = trainer.stopper
    trainer.stopper = StrictEarlyStopping(patience=trainer.args.patience)
    trainer.stopper.best_fitness = restored.best_fitness
    trainer.stopper.best_epoch = restored.best_epoch
    if trainer.start_epoch:
        print(
            f"이어 학습: {trainer.start_epoch + 1}에폭부터, "
            f"지금까지 최고 {restored.best_fitness:.5f} (epoch {restored.best_epoch})"
        )
    info = {
        "optimizer": type(trainer.optimizer).__name__,
        "lr0": trainer.args.lr0,
        "group_initial_lrs": [
            float(group.get("initial_lr", group["lr"])) for group in trainer.optimizer.param_groups
        ],
        "trainable_parameters": sum(
            parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad
        ),
        "frozen_parameters": sum(
            parameter.numel()
            for parameter in trainer.model.parameters()
            if not parameter.requires_grad
        ),
    }
    write_json(Path(trainer.save_dir) / "optimizer_used.json", info)
    print(json.dumps(info, ensure_ascii=False))


def log_epoch(trainer: BaselineTrainer) -> None:
    """매 에폭 train/val 손실과 검증 지표를 함께 출력한다."""
    if not trainer.validator.training:
        return
    values = {
        "epoch": trainer.epoch + 1,
        **trainer.label_loss_items(trainer.tloss, prefix="train"),
        **{key: float(value) for key, value in trainer.metrics.items()},
    }
    print(json.dumps(values, ensure_ascii=False))

## 3-2. bbox copy-paste 증강

ultralytics의 `copy_paste`는 segmentation 라벨이 있어야 동작해서 bbox만 있는 우리 데이터에서는 아무것도
하지 않습니다. 여기서는 학습 사진의 **정답 bbox를 그대로 잘라** 다른 학습 사진의 빈 배경에 붙입니다.
4절 학습기(`BaselineTrainer.build_dataset`)가 학습 데이터셋에만 연결하고, 검증·평가 데이터셋에는 쓰지 않습니다.

1. **crop 묶음**: 학습 시작 때 학습 목록(대회 train + AI Hub, validation 제외)의 정답 bbox를 잘라
   사진을 읽을 때와 같은 크기(긴 변 640)로 메모리에 둡니다. 다른 bbox와 겹치는 bbox는 옆 알약 조각이
   라벨 없이 함께 붙으므로 뺍니다. AI Hub를 더하면 수천 장을 한 번 읽어 시작이 1~2분 늦고 RAM을 약 1GB 씁니다.
2. **붙이기**: 사진(모자이크 타일)을 읽을 때마다 `COPY_PASTE_PROB` 확률로 1~`COPY_PASTE_MAX`개를 붙입니다.
   클래스를 균등하게 먼저 고르고(외부 전용 클래스 포함) 그 안에서 crop을 고릅니다. crop 전체(여백 포함)가
   기존 정답·먼저 붙인 crop과 겹치지 않는 위치를 30번까지 찾고, 못 찾으면 붙이지 않습니다.
   붙인 bbox 라벨은 정수로 자르며 생긴 여백을 빼고 원래 bbox의 소수점 좌표를 씁니다.
3. 모자이크·회전·색 증강보다 먼저 붙이므로 붙인 알약도 같은 증강을 받습니다. 가장자리를 흐리게 섞으면
   bbox 경계가 흐려져 mAP75-95에 불리하므로 그대로 붙입니다.
4. ultralytics가 마지막 `close_mosaic`(10)에폭에 모자이크·copy_paste를 끄는 것과 같이 함께 끕니다.


In [ ]:
PLACE_TRIES = 30  # 겹치지 않는 위치를 찾는 시도 횟수. 못 찾으면 그 crop은 붙이지 않는다.


def xywhn_to_xyxy(bboxes: np.ndarray, width: int, height: int) -> np.ndarray:
    """정규화 xywh(중심) bbox를 픽셀 xyxy로 바꾼다."""
    centers = bboxes[:, :2] * [width, height]
    sizes = bboxes[:, 2:] * [width, height]
    return np.concatenate([centers - sizes / 2, centers + sizes / 2], axis=1)


def overlaps_any(box: np.ndarray, boxes: np.ndarray) -> bool:
    """box(xyxy)가 boxes 중 하나와 넓이가 있는 영역으로 겹치는지."""
    if not len(boxes):
        return False
    return bool(
        (
            (boxes[:, 0] < box[2])
            & (boxes[:, 2] > box[0])
            & (boxes[:, 1] < box[3])
            & (boxes[:, 3] > box[1])
        ).any()
    )


class BBoxCopyPasteDataset(YOLODataset):
    """학습 데이터셋에 bbox copy-paste를 더한 YOLODataset.

    `enable()`로 이미 만든 학습 데이터셋을 바꿔 쓴다. 모자이크도 같은 객체의
    `get_image_and_label()`로 타일을 읽으므로 모자이크 타일마다 따로 붙는다.
    """

    @classmethod
    def enable(
        cls,
        dataset: YOLODataset,
        probability: float,
        max_paste: int,
        classes: list[int] | None = None,
    ) -> "BBoxCopyPasteDataset":
        """학습 데이터셋을 이 클래스로 바꾸고 그 데이터셋의 정답 bbox로 crop 묶음을 만든다.

        probability는 사진(모자이크 타일)마다 붙일 확률, max_paste는 한 번에 붙일 최대 개수다.
        classes가 None이면 모든 클래스를, 아니면 그 YOLO 클래스 번호만 붙인다. 붙일 crop은
        클래스를 균등하게 먼저 고르고 그 안에서 고른다.
        """
        if type(dataset) is not YOLODataset or not dataset.augment:
            raise TypeError("증강을 켠 학습용 YOLODataset에만 적용할 수 있습니다.")
        if not 0.0 <= probability <= 1.0 or max_paste < 1:
            raise ValueError("확률은 0~1, 최대 개수는 1 이상이어야 합니다.")
        dataset.__class__ = cls
        dataset.copy_paste_probability = probability
        dataset.copy_paste_max = max_paste
        dataset.build_crop_bank(None if classes is None else set(classes))
        return dataset

    def build_crop_bank(self, classes: set[int] | None) -> None:
        """다른 정답과 겹치지 않는 bbox만 잘라 load_image와 같은 크기(긴 변 imgsz)로 보관한다.

        다른 bbox와 겹치는 crop은 옆 알약 조각이 라벨 없이 함께 붙으므로 뺀다.
        """
        with ThreadPool(NUM_THREADS) as pool:
            results = pool.map(lambda label: self.crop_image(label, classes), self.labels)
        self.crop_bank = [crop for crops in results for crop in crops]
        if not self.crop_bank:
            raise ValueError("copy-paste에 쓸 crop이 없습니다.")
        by_class: dict[int, list[int]] = {}
        for index, crop in enumerate(self.crop_bank):
            by_class.setdefault(crop["cls"], []).append(index)
        self.crop_classes = sorted(by_class)
        self.crop_index = by_class
        n_boxes = sum(len(label["cls"]) for label in self.labels)
        gigabytes = sum(crop["img"].nbytes for crop in self.crop_bank) / 1024**3
        LOGGER.info(
            f"bbox copy-paste: crop {len(self.crop_bank)}개 / 정답 {n_boxes}개, "
            f"클래스 {len(self.crop_classes)}종, 메모리 {gigabytes:.2f}GB, "
            f"확률 {self.copy_paste_probability:g}, 최대 {self.copy_paste_max}개"
        )

    def crop_image(self, label: dict, classes: set[int] | None) -> list[dict]:
        """사진 한 장에서 쓸 수 있는 bbox crop을 자른다."""
        height, width = label["shape"]
        boxes = xywhn_to_xyxy(label["bboxes"], width, height)
        keep = []
        for index, (box, class_index) in enumerate(zip(boxes, label["cls"][:, 0], strict=True)):
            others = np.delete(boxes, index, axis=0)
            if (classes is None or int(class_index) in classes) and not overlaps_any(box, others):
                keep.append(index)
        if not keep:
            return []
        image = cv2.imread(label["im_file"])
        if image is None:
            raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {label['im_file']}")
        if image.shape[:2] != (height, width):
            raise ValueError(f"라벨과 실제 이미지 크기가 다릅니다: {label['im_file']}")
        ratio = self.imgsz / max(height, width)  # load_image의 긴 변 맞춤과 같은 비율
        crops = []
        for index in keep:
            x1, y1, x2, y2 = boxes[index].clip(0, [width, height, width, height])
            left, top = math.floor(x1), math.floor(y1)
            right, bottom = math.ceil(x2), math.ceil(y2)
            if right - left < 2 or bottom - top < 2:
                continue
            crop = image[top:bottom, left:right]
            size = (max(1, round((right - left) * ratio)), max(1, round((bottom - top) * ratio)))
            if ratio != 1:
                crop = cv2.resize(crop, size, interpolation=cv2.INTER_LINEAR)
            # crop 안에서 원래 bbox의 소수점 좌표. 정수로 자르며 생긴 여백을 라벨에 넣지 않는다.
            scale_x, scale_y = size[0] / (right - left), size[1] / (bottom - top)
            inner = np.array(
                [
                    (x1 - left) * scale_x,
                    (y1 - top) * scale_y,
                    (x2 - left) * scale_x,
                    (y2 - top) * scale_y,
                ]
            )
            crops.append(
                {
                    "img": np.ascontiguousarray(crop),
                    "inner": inner,
                    "cls": int(label["cls"][index, 0]),
                    "ratio": ratio,
                }
            )
        return crops

    def close_mosaic(self, hyp: object) -> None:
        """ultralytics가 마지막 close_mosaic 에폭에 copy_paste를 끄는 것과 같이 함께 끈다."""
        self.copy_paste_probability = 0.0
        super().close_mosaic(hyp)

    def update_labels_info(self, label: dict) -> dict:
        """읽은 사진에 확률적으로 crop을 붙이고 라벨을 더한 뒤 ultralytics 형식으로 바꾼다."""
        if self.copy_paste_probability and random.random() < self.copy_paste_probability:
            self.paste_crops(label)
        return super().update_labels_info(label)

    def paste_crops(self, label: dict) -> None:
        """기존 정답·이미 붙인 crop과 겹치지 않는 배경 위치에 1~최대 개수의 crop을 붙인다."""
        if label["bbox_format"] != "xywh" or not label["normalized"]:
            raise ValueError("정규화 xywh 라벨만 지원합니다.")
        # 읽은 사진은 모자이크 버퍼에 그대로 남으므로 복사본에 붙인다.
        image = label["img"].copy()
        height, width = image.shape[:2]
        occupied = xywhn_to_xyxy(label["bboxes"], width, height)
        # 원본 해상도가 crop을 자른 사진과 다르면 실제 크기가 같도록 맞춘다.
        target_ratio = height / label["ori_shape"][0]
        new_boxes, new_cls = [], []
        for _ in range(random.randint(1, self.copy_paste_max)):
            class_index = random.choice(self.crop_classes)
            crop = self.crop_bank[random.choice(self.crop_index[class_index])]
            patch, inner = crop["img"], crop["inner"]
            scale = target_ratio / crop["ratio"]
            if abs(scale - 1) > 0.01:
                size = (
                    max(1, round(patch.shape[1] * scale)),
                    max(1, round(patch.shape[0] * scale)),
                )
                scale_x, scale_y = size[0] / patch.shape[1], size[1] / patch.shape[0]
                inner = inner * np.array([scale_x, scale_y, scale_x, scale_y])
                patch = cv2.resize(patch, size, interpolation=cv2.INTER_LINEAR)
            patch_height, patch_width = patch.shape[:2]
            if patch_width > width or patch_height > height:
                continue
            for _ in range(PLACE_TRIES):
                left = random.randint(0, width - patch_width)
                top = random.randint(0, height - patch_height)
                # crop 전체(여백 포함)가 다른 알약을 가리지 않아야 한다.
                area = np.array([left, top, left + patch_width, top + patch_height], dtype=float)
                if overlaps_any(area, occupied):
                    continue
                image[top : top + patch_height, left : left + patch_width] = patch
                x1, y1, x2, y2 = inner + [left, top, left, top]
                new_boxes.append(
                    [
                        (x1 + x2) / 2 / width,
                        (y1 + y2) / 2 / height,
                        (x2 - x1) / width,
                        (y2 - y1) / height,
                    ]
                )
                new_cls.append(class_index)
                occupied = np.vstack([occupied, area])
                break
        if not new_boxes:
            return
        label["img"] = image
        label["bboxes"] = np.concatenate(
            [label["bboxes"], np.array(new_boxes, dtype=label["bboxes"].dtype)]
        )
        label["cls"] = np.concatenate(
            [label["cls"], np.array(new_cls, dtype=label["cls"].dtype).reshape(-1, 1)]
        )

## 4. 단일 holdout 학습과 에폭별 꺾은선 그래프

이 셀에서 실제 학습이 한 번 시작됩니다(SMOKE=True면 동작 확인). 교차검증 반복 학습은 하지 않습니다.
동일한 설정으로 완료한 결과는 재사용하고, 중간에 끊긴 결과(`train/weights/last.pt`가 있음)는
그 체크포인트에서 이어 학습합니다(옵티마이저·EMA·학습률 스케줄·조기 중단 카운트 포함).
끊긴 결과를 덮어쓰지는 않습니다.
OOM이 발생해도 batch=16을 임의로 줄이지 않습니다.

학습 데이터는 2-2절의 `TRAIN_DATA_YAML`(대회 train + 외부 사진, 외부 전용 클래스 포함)이고,
학습 중 검증과 5절 이후 평가는 대회 validation만 씁니다.
외부 사진이 늘어난 만큼 에폭당 시간이 늘어납니다(2-2절의 `train 배수`).
bbox copy-paste는 학습 데이터셋을 만들 때 crop 묶음을 먼저 만들고(로그의 `bbox copy-paste: crop ...개`),
학습 중 검증·5절 이후 평가에는 적용하지 않습니다.

best.pt는 고정한 validation의 mAP75–95가 가장 높은 체크포인트입니다.
학습 뒤 그리는 그림은 yolo11m 노트북(`exp/yolo11m` 브랜치 9-1·9-2)과 같은 기준입니다.

- **상세 2×5** (`training_curves.png`): 윗줄은 loss 세 가지(실선 train, 점선 val. YOLO26은
  `box`·`cls`·`l1`, YOLO11은 `box`·`cls`·`dfl`), 에폭당 시간(학습 + 검증 + 체크포인트 저장),
  학습률입니다. 아랫줄은 Precision·Recall·mAP50·mAP50-95·mAP75-95입니다.
  회색 점선은 best epoch(검증 mAP75-95 최고), 빨간 별은 best.pt를 고른 지점입니다.
- **대시보드 2×3** (`training_dashboard.png`): Train loss·Validation loss(각각 loss 성분 합),
  mAP75-95, Recall, Precision, 매 에폭 검증의 이미지당 추론시간(ms)입니다.

에폭 시간(`train/epoch_times.csv`)과 에폭별 추론시간(`results.csv`의 `speed/inference_ms`)은
이 기록을 추가한 뒤 학습한 결과부터 있습니다. 없으면 그 칸에 기록 없음으로 표시합니다.

In [ ]:
def train_baseline() -> Path:
    """단일 holdout으로 사전학습 가중치부터 한 번 학습하고 best.pt 경로를 반환한다.

    끊긴 학습(train/weights/last.pt가 있음)은 그 체크포인트에서 이어 학습한다.
    """
    train_dir = EXPERIMENT_DIR / "train"
    complete_path = train_dir / "completed.json"
    last = train_dir / "weights" / "last.pt"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text(encoding="utf-8"))
        # 다른 PC·Colab에서 만든 기록이면 경로가 달라도 같은 실험 폴더의 best.pt를 쓴다.
        best = train_dir / "weights" / Path(complete["best_pt"]).name
        if training_config_diff(complete["config"]) or not best.is_file():
            raise ValueError("기존 완료 기록과 설정/가중치가 다릅니다.")
        print(f"완료된 단일 학습 사용: {best}")
        return best

    def mark_complete(best: Path) -> Path:
        if not best.is_file():
            raise FileNotFoundError(f"best.pt가 없습니다: {best}")
        write_json(
            complete_path,
            {
                "best_pt": str(best),
                "config": EXPERIMENT_CONFIG,
                "pretrained_sha256": file_sha256(PRETRAINED_PATH),
            },
        )
        return best

    resume = train_dir.exists()
    if resume and not last.is_file():
        raise FileExistsError(
            f"last.pt 없이 끊긴 결과가 있습니다(첫 에폭 전): {train_dir}. "
            "새 실험 설정 셀부터 실행하세요."
        )
    if resume and is_finished(last):
        print(f"학습은 끝났고 완료 기록만 없습니다. 기록을 남깁니다: {train_dir}")
        return mark_complete(train_dir / "weights" / "best.pt")
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAINED_PATH.is_file() and (PROJECT_ROOT / MODEL_NAME).is_file():
        shutil.copy2(PROJECT_ROOT / MODEL_NAME, PRETRAINED_PATH)
    init_seeds(SEED, deterministic=True)
    model = YOLO(str(last if resume else PRETRAINED_PATH))
    model.to(DEVICE)
    model.add_callback("on_train_start", start_training)
    model.add_callback("on_fit_epoch_end", log_epoch)
    epoch_timer = EpochTimer()
    model.add_callback("on_train_start", epoch_timer.on_train_start)
    model.add_callback("on_train_epoch_start", epoch_timer.on_epoch_start)
    model.add_callback("on_fit_epoch_end", epoch_timer.on_epoch_end)
    args = {
        **AUGMENTATION,
        "data": str(TRAIN_DATA_YAML),
        "epochs": TRAIN_EPOCHS,
        "patience": PATIENCE,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "seed": SEED,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "freeze": FREEZE,
        "device": DEVICE,
        "workers": TRAIN_WORKERS,
        "deterministic": True,
        "val": True,
        "save": True,
        "plots": True,
        "amp": True,  # MPS에서는 ultralytics가 자동으로 끈다.
        "fraction": TRAIN_FRACTION,
        "cos_lr": False,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "max_det": MAX_DET,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "project": str(train_dir.parent),
        "name": train_dir.name,
        "exist_ok": False,
    }
    if resume:
        # 학습 인자는 last.pt에 저장된 값을 그대로 쓴다. 바뀔 수 있는 데이터 경로·장치·
        # worker 수·결과 폴더만 지금 값으로 넘긴다(ultralytics check_resume이 허용하는 항목).
        print(f"끊긴 학습을 이어 합니다: {last}")
        args = {
            "resume": True,
            "data": str(TRAIN_DATA_YAML),
            "device": DEVICE,
            "workers": TRAIN_WORKERS,
            "save_dir": str(train_dir),
        }
    try:
        model.train(trainer=BaselineTrainer, **args)
        return mark_complete(Path(model.trainer.best))
    finally:
        del model
        gc.collect()
        empty_device_cache()


print("===== combo 단일 holdout | 교차검증 없이 한 번 학습 =====")
BEST_WEIGHTS = train_baseline()
print(f"best.pt: {BEST_WEIGHTS}")

In [ ]:
def read_history(train_dir: Path) -> pd.DataFrame:
    """학습 폴더의 results.csv를 읽어 열 이름 공백을 지운다."""
    history = pd.read_csv(train_dir / "results.csv")
    history.columns = history.columns.str.strip()
    return history.reset_index(drop=True)


def best_epoch_row(history: pd.DataFrame) -> pd.Series:
    """best.pt를 저장한 에폭 행(mAP75-95가 이전 최고보다 클 때만 갱신하므로 첫 최댓값)."""
    return history.loc[history[MAIN_METRIC_KEY].idxmax()]


def plot_training_curves(train_dir: Path, output_path: Path) -> None:
    """yolo11m 9-1과 같은 2×5 그림. 회색 점선은 best epoch, 빨간 별은 best 점수다."""
    curves = read_history(train_dir)
    # loss 이름은 모델마다 달라(YOLO26 box·cls·l1, YOLO11 box·cls·dfl) 기록된 열을 쓴다.
    loss_names = [
        column.split("/", 1)[1]
        for column in curves
        if column.startswith("train/") and f"val/{column.split('/', 1)[1]}" in curves
    ]
    metric_keys = {
        "Precision": "metrics/precision(B)",
        "Recall": "metrics/recall(B)",
        "mAP50": "metrics/mAP50(B)",
        "mAP50-95": "metrics/mAP50-95(B)",
        "mAP75-95": MAIN_METRIC_KEY,
    }
    epoch = curves["epoch"]
    best = best_epoch_row(curves)
    best_epoch, best_score = int(best["epoch"]), float(best[MAIN_METRIC_KEY])
    fig, axes = plt.subplots(2, 5, figsize=(25, 9))
    for ax, name in zip(axes[0, :3], loss_names, strict=False):
        ax.plot(epoch, curves[f"train/{name}"], marker=".", label="train")
        ax.plot(epoch, curves[f"val/{name}"], marker=".", linestyle="--", label="val")
        ax.set_title(name)

    ax = axes[0, 3]
    times_path = train_dir / "epoch_times.csv"
    if times_path.is_file():
        epoch_times = pd.read_csv(times_path)
        ax.bar(epoch_times["epoch"], epoch_times["epoch_seconds"], color="lightgray")
        ax.set_title(f"seconds per epoch (mean {epoch_times['epoch_seconds'].mean():.0f}s)")
    else:
        ax.text(
            0.5,
            0.5,
            "No per-epoch data recorded",
            transform=ax.transAxes,
            ha="center",
            va="center",
            color="dimgray",
        )
        ax.set_title("seconds per epoch")

    ax = axes[0, 4]
    # 파라미터 묶음(lr/pg0, pg1, ...)마다 한 줄. 범례 없이 높이만 본다
    for col in [c for c in curves.columns if c.startswith("lr/")]:
        ax.plot(epoch, curves[col], color="tab:purple", alpha=0.6)
    ax.set_title("learning rate (param groups)")

    for ax, (title, key) in zip(axes[1], metric_keys.items(), strict=True):
        ax.plot(epoch, curves[key], marker=".")
        ax.set_title(title)
    axes[1, 4].scatter(best_epoch, best_score, marker="*", s=300, color="red", zorder=3)
    axes[1, 4].annotate(
        f"best {best_score:.4f}",
        (best_epoch, best_score),
        xytext=(5, -15),
        textcoords="offset points",
    )

    for ax in axes.flat:
        ax.axvline(best_epoch, color="gray", linestyle=":")
        ax.set_xlabel("epoch")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    fig.suptitle("YOLO26m / holdout")
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


TRAIN_DIR = EXPERIMENT_DIR / "train"
plot_training_curves(TRAIN_DIR, EXPERIMENT_DIR / "training_curves.png")
best = best_epoch_row(read_history(TRAIN_DIR))
print(
    f"best epoch {int(best['epoch'])} / 전체 {int(read_history(TRAIN_DIR)['epoch'].max())}에폭 | "
    f"mAP75-95 {best[MAIN_METRIC_KEY]:.5f}, "
    f"P {best['metrics/precision(B)']:.4f}, R {best['metrics/recall(B)']:.4f}"
)

In [ ]:
def plot_training_dashboard(train_dir: Path, output_path: Path, label: str = "holdout") -> None:
    """yolo11m 9-2와 같은 2×3 그림. loss는 각 loss 성분의 합이다."""
    panels = [
        ("train_total", "Train loss", "Loss"),
        ("val_total", "Validation loss", "Loss"),
        (MAIN_METRIC_KEY, "mAP75-95", "mAP@[0.75:0.95]"),
        ("metrics/recall(B)", "Recall", "Recall"),
        ("metrics/precision(B)", "Precision", "Precision"),
        (INFERENCE_METRIC_KEY, "Inference time (lower is faster)", "ms / image"),
    ]
    history = read_history(train_dir)
    history = history.apply(pd.to_numeric, errors="coerce").dropna(subset=["epoch"])
    history = history.sort_values("epoch").drop_duplicates("epoch", keep="last")
    if history.empty:
        raise ValueError(f"유효한 에폭 기록이 없습니다: {train_dir / 'results.csv'}")
    for split in ("train", "val"):
        loss_columns = [
            column
            for column in history
            if column.startswith(f"{split}/") and column.endswith("_loss")
        ]
        history[f"{split}_total"] = (
            history[loss_columns].sum(axis=1, min_count=len(loss_columns))
            if loss_columns
            else pd.Series(float("nan"), index=history.index)
        )
    color = plt.get_cmap("tab10")(0)

    with plt.rc_context({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 11}):
        fig, axes = plt.subplots(2, 3, figsize=(18, 9.5), sharex=True)
        for ax, (key, title, ylabel) in zip(axes.flat, panels, strict=True):
            if key in history and history[key].notna().any():
                ax.plot(
                    history["epoch"],
                    history[key],
                    color=color,
                    linewidth=2,
                    marker="o",
                    markersize=3,
                    label=label,
                )
            else:
                ax.text(
                    0.5,
                    0.5,
                    "No per-epoch data recorded",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    color="dimgray",
                )
            ax.set_title(title, fontweight="bold", pad=10)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(ylabel)
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(True, alpha=0.25)
            ax.spines[["top", "right"]].set_visible(False)
            if key.startswith("metrics/"):
                ax.set_ylim(0, 1.05)
            else:
                ax.set_ylim(bottom=0)
        handle = Line2D([0], [0], color=color, linewidth=2, marker="o", markersize=4, label=label)
        fig.suptitle(
            "YOLO26m | Training & validation overview", fontsize=19, fontweight="bold", y=0.99
        )
        fig.legend(handles=[handle], loc="upper center", bbox_to_anchor=(0.5, 0.955), frameon=False)
        fig.tight_layout(rect=(0, 0, 1, 0.90), h_pad=2.5, w_pad=2.0)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    if INFERENCE_METRIC_KEY not in history or not history[INFERENCE_METRIC_KEY].notna().any():
        print("에폭별 추론시간 기록 없음. 기록을 추가한 뒤 학습한 결과부터 그려집니다.")
    print(f"그래프 저장: {output_path}")


plot_training_dashboard(TRAIN_DIR, EXPERIMENT_DIR / "training_dashboard.png")

## 5. best.pt 평가 · 두 신뢰도의 혼동행렬 · 클래스별 AP

단일 학습의 같은 best.pt와 고정 validation을 사용하고 conf만 0.001/0.25로 바꿉니다.
두 평가의 원시·정규화 혼동행렬을 저장하고 같은 색상 범위로 비교합니다.
주 성능은 conf=0.001이며 검증에 없는 클래스 AP는 NaN으로 둡니다.
교차검증이나 fold 간 평균·표준편차 계산은 하지 않습니다.
모델 클래스 수에 맞춰 `TRAIN_DATA_YAML`로 평가합니다. validation 목록은 대회 사진 그대로이고,
외부 전용 클래스 예측은 검증기에서 지우므로 혼동행렬은 대회 56종 + background만 남깁니다.


In [ ]:
def per_class_metrics(metrics: object) -> pd.DataFrame:
    """검증에 없는 클래스는 NaN으로 두고 클래스별 AP·학습/검증 수를 만든다."""
    table = CLASS_MAP.copy()
    table["group"] = table["category_id"].map(LABELS["category_id"].value_counts().map(size_group))
    mapping = SPLIT_TABLE.set_index("file_name")["split"]
    box_split = LABELS["file_name"].map(mapping)
    for split in ("train", "val"):
        counts = LABELS.loc[box_split == split, "category_id"].value_counts()
        table[f"n_{split}_bbox"] = table["category_id"].map(counts).fillna(0).astype(int)
    aihub_counts = AIHUB_LABELS["category_id"].value_counts()
    table["n_aihub_bbox"] = table["category_id"].map(aihub_counts).fillna(0).astype(int)
    box = metrics.box
    scores = pd.DataFrame(
        {
            "ap50": box.ap50,
            "ap50_95": box.ap,
            "ap75_95": ap75_95(box.all_ap) if len(box.all_ap) else [],
            "precision_max_f1": box.p,
            "recall_max_f1": box.r,
        },
        index=box.ap_class_index,
    )
    table = table.join(scores, on="yolo_idx")
    present = table["n_val_bbox"] > 0
    table.loc[present, scores.columns] = table.loc[present, scores.columns].fillna(0.0)
    table.loc[table["n_val_bbox"] == 0, scores.columns] = np.nan
    return table


def compare_confusion_matrices(matrices: dict[float, np.ndarray], output_path: Path) -> None:
    """동일한 색상 범위로 두 혼동행렬을 나란히 저장한다."""
    names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    maximum = max(1.0, max(float(matrix.max()) for matrix in matrices.values()))
    fig, axes = plt.subplots(1, len(CM_CONFS), figsize=(24, 11), squeeze=False)
    for axis, confidence in zip(axes[0], CM_CONFS, strict=True):
        mesh = axis.imshow(matrices[confidence], cmap="Blues", vmin=0, vmax=maximum)
        axis.set(
            title=f"conf={confidence:g}, match IoU={CM_MATCH_IOU}",
            xlabel="True",
            ylabel="Predicted",
            xticks=range(len(names)),
            yticks=range(len(names)),
            xticklabels=names,
            yticklabels=names,
        )
        axis.tick_params(axis="x", labelrotation=90, labelsize=6)
        axis.tick_params(axis="y", labelsize=6)
    fig.colorbar(mesh, ax=axes.ravel().tolist(), label="Count", fraction=0.02, pad=0.02)
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.close(fig)


def evaluate_baseline(weights: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """고정 validation을 두 신뢰도로 평가하고 지표·행렬·실제 평가 예측을 저장한다."""
    out_dir = EXPERIMENT_DIR / "evaluation"
    matrices, rows = {}, []
    main_per_class = None
    class_names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    class_index = dict(zip(CLASS_MAP["category_id"], CLASS_MAP["yolo_idx"], strict=True))
    expected_files = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    for confidence in CM_CONFS:
        model = YOLO(str(weights))
        try:
            metrics = model.val(
                validator=ReadOnlyValidator,
                data=str(TRAIN_DATA_YAML),
                split="val",
                imgsz=IMGSZ,
                batch=BATCH,
                device=DEVICE,
                workers=WORKERS,
                conf=confidence,
                iou=NMS_IOU,
                max_det=MAX_DET,
                nms=NMS_ENABLED,
                agnostic_nms=False,
                quantize=32,
                plots=True,
                augment=False,
                save_json=False,
                project=str(out_dir),
                name=f"conf_{confidence:g}",
                exist_ok=True,
            )
            predictions = metrics.validation_predictions
            if (
                len(predictions) != len(expected_files)
                or {record["file_name"] for record in predictions} != expected_files
            ):
                raise AssertionError("평가 예측 이미지와 고정 validation이 다릅니다.")
            write_json(out_dir / f"predictions_conf_{confidence:g}.json", predictions)
            full_matrix = metrics.confusion_matrix.matrix
            keep = [*range(N_COMP_CLASSES), full_matrix.shape[0] - 1]  # 대회 클래스 + background
            matrix = full_matrix[np.ix_(keep, keep)].copy()
            if matrix.sum() != full_matrix.sum():
                raise AssertionError("외부 전용 클래스가 혼동행렬에 남았습니다.")
            matrices[confidence] = matrix
            pd.DataFrame(matrix.astype(int), index=class_names, columns=class_names).to_csv(
                out_dir / f"confusion_matrix_conf_{confidence:g}.csv", index_label="predicted/true"
            )
            true_positives = float(np.trace(matrix[:-1, :-1]))
            predicted = float(matrix[:-1, :].sum())
            targets = float(matrix[:, :-1].sum())
            table = per_class_metrics(metrics)
            if confidence == EVAL_CONF:
                main_per_class = table
                table.to_csv(out_dir / "per_class.csv", index=False)
            rows.append(
                {
                    "evaluation": "holdout",
                    "conf": confidence,
                    MAIN_METRIC: map75_95(metrics.box),
                    "precision_macro_max_f1_iou0.5": float(metrics.box.mp),
                    "recall_macro_max_f1_iou0.5": float(metrics.box.mr),
                    "precision_micro_fixed_conf_cm": true_positives / predicted
                    if predicted
                    else 0.0,
                    "recall_micro_fixed_conf_cm": true_positives / targets if targets else 0.0,
                    "cm_match_iou": CM_MATCH_IOU,
                    "n_predictions": int(predicted),
                    "n_ground_truths": int(targets),
                    "classes_not_in_val": int(table["n_val_bbox"].eq(0).sum()),
                    "val_only_classes": int(
                        ((table["n_train_bbox"] == 0) & (table["n_val_bbox"] > 0)).sum()
                    ),
                    **{
                        f"ap75_95_group_{group}": float(
                            table.loc[table["group"] == group, "ap75_95"].mean()
                        )
                        for group in ("21+", "6-20", "≤5")
                    },
                    **{
                        f"confuse_{first}_{second}": int(
                            matrix[class_index[first], class_index[second]]
                            + matrix[class_index[second], class_index[first]]
                        )
                        for first, second in CONFUSION_PAIRS
                        if first in class_index and second in class_index
                    },
                    "aihub_train_images": int(AIHUB_INFO.get("train_images", 0)),
                    "aihub_train_bbox": len(AIHUB_LABELS),
                    "aihub_extra_classes": int(AIHUB_INFO.get("extra_classes", 0)),
                    "validation_batch": BATCH,
                    "val_forward_ms_per_image": float(metrics.speed["inference"]),
                    "nms_iou": NMS_IOU,
                    "max_det": MAX_DET,
                }
            )
        finally:
            del model
            gc.collect()
            empty_device_cache()
    comparison = out_dir / "confusion_matrix_comparison.png"
    compare_confusion_matrices(matrices, comparison)
    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "threshold_comparison.csv", index=False)
    display(result)
    display(Image(filename=str(comparison)))
    return result, main_per_class

In [ ]:
THRESHOLD_RESULTS, PER_CLASS_RESULTS = evaluate_baseline(BEST_WEIGHTS)
THRESHOLD_RESULTS.to_csv(EXPERIMENT_DIR / "threshold_comparison.csv", index=False)
PER_CLASS_RESULTS.to_csv(EXPERIMENT_DIR / "per_class.csv", index=False)
PRIMARY_RESULTS = THRESHOLD_RESULTS.loc[THRESHOLD_RESULTS["conf"] == EVAL_CONF].copy()
metric_columns = [
    MAIN_METRIC,
    "precision_macro_max_f1_iou0.5",
    "recall_macro_max_f1_iou0.5",
    "precision_micro_fixed_conf_cm",
    "recall_micro_fixed_conf_cm",
    "ap75_95_group_21+",
    "ap75_95_group_6-20",
    "ap75_95_group_≤5",
]
SUMMARY = PRIMARY_RESULTS.set_index("evaluation")[metric_columns].T
SUMMARY.to_csv(EXPERIMENT_DIR / "summary_holdout.csv")
display(PRIMARY_RESULTS)
display(SUMMARY)
class_summary = PER_CLASS_RESULTS.set_index(["category_id", "dl_name", "group"])[
    [
        "n_train_bbox",
        "n_aihub_bbox",
        "n_val_bbox",
        "ap75_95",
        "precision_max_f1",
        "recall_max_f1",
    ]
]
class_summary.to_csv(EXPERIMENT_DIR / "per_class_summary_holdout.csv")
display(class_summary)
print("고정 validation 한 개를 평가했습니다. 검증에 없는 클래스 AP는 평가 불가(NaN)입니다.")

## 5-2. 각인 OCR 학습과 클래스 재분류

알약에 새겨진 식별문자(각인)를 읽는 작은 OCR 모델을 **YOLO와 같은 학습 사진으로 따로 학습**하고,
탐지 bbox의 클래스를 OCR로 다시 고릅니다. YOLO 가중치와 학습은 위 4절 그대로입니다. `USE_OCR = False`면 건너뜁니다.

1. **정답 텍스트**: bbox마다 원본 JSON의 `drug_dir`(이 사진에서 보이는 면)에 따라 `print_front` 또는
   `print_back`을 씁니다. 설명어는 한 글자로 바꿉니다(`십자분할선`→`+`, `분할선`→`|`, `마크`→`@`).
   공백은 지우고 영문은 대문자로 맞춥니다. 예: `I분할선Y` → `I|Y`, `마크 PGN    150` → `@PGN150`.
   면을 모르는 bbox(대회 보정에서 추가한 bbox)는 OCR 학습에서 뺍니다.
2. **학습 데이터**: 학습 목록(대회 train + AI Hub)의 정답 bbox crop입니다. validation 사진은 쓰지 않습니다.
   bbox 중심에서 긴 변의 1.3배 정사각형을 잘라, 학습 때마다 임의 각도로 돌리고 bbox 크기·위치를 ±10%·±5% 흔들고
   밝기·대비를 바꿉니다. 알약이 아무 방향으로 놓이므로 회전을 학습합니다.
3. **모델**: CRNN(VGG형 CNN + 2층 BiLSTM)을 처음부터 CTC loss로 학습합니다(AdamW, 코사인 학습률).
   best는 대회 validation 정답 crop의 **각인 분류 정확도**(아래 4로 고른 클래스가 정답과 같은 비율)가 가장 높은 에폭입니다.
4. **클래스 점수**: crop을 `OCR_ROTATIONS` 방향으로 돌려 보고, 모델 클래스(외부 전용 포함)마다 앞·뒷면 각인이
   나올 CTC 로그 확률 중 가장 큰 값을 그 클래스 점수로 씁니다. 이것을 `OCR_TEMPERATURE`로 나눠 softmax한 것이 OCR 확률입니다.
   각인이 없는 면은 빈 문자열이라 "글자가 안 보임"도 하나의 후보가 됩니다. 각인이 같은 약끼리는 확률이 나뉩니다.
5. **재분류**: 탐지 bbox마다 OCR이 가리키는 클래스 확률이 `OCR_SWITCH_PROB` 이상이고, YOLO가 고른 클래스의
   OCR 확률이 `OCR_KEEP_PROB` 이하일 때만 클래스를 바꿉니다. 박스 좌표와 점수는 YOLO 값을 그대로 둡니다.
   외부 전용 클래스로 바뀐 예측은 평가에서 지웁니다(5절과 같은 규칙).

평가는 5절에서 저장한 conf=0.001 예측(`evaluation/predictions_conf_0.001.json`)을 그대로 쓰고, ultralytics와
같은 매칭으로 OCR 전·후 mAP75-95를 계산합니다. OCR 전 값이 5절 값과 거의 같은지 함께 출력합니다.
결과는 실험 폴더의 `ocr/`(가중치·에폭 기록·그래프·검증 재분류 표)에 남습니다.


In [ ]:
OCR_DIR = EXPERIMENT_DIR / "ocr"
# 각인 설명어 → 한 글자 토큰. 긴 말부터 바꾼다(십자분할선 안의 분할선).
OCR_TOKENS = {"십자분할선": "+", "분할선": "|", "마크": "@"}
FACE_COLUMNS = {"앞면": "print_front", "뒷면": "print_back"}
OCR_CROP_MARGIN = 1.3  # crop은 bbox 긴 변의 이 배수 정사각형. 회전·흔들기에 쓸 여백이다
OCR_SEQ_LEN = OCR_IMGSZ // 4  # CRNN 출력 길이(가로 1/4)
OCR_SCORE_BATCH = 64  # 클래스 점수를 계산할 때 한 번에 넣는 crop 수(회전 수만큼 늘어남)
IOUV = np.linspace(0.5, 0.95, 10)  # ultralytics 검증과 같은 IoU 10단계
# OCR 학습 결과에 영향을 주는 설정. 저장된 OCR 기록과 다르면 다시 학습한다.
OCR_TRAINING_KEYS = [
    "ocr_imgsz",
    "ocr_epochs",
    "ocr_batch",
    "ocr_lr",
    "ocr_patience",
    "fraction",
    "seed",
]


def normalize_imprint(text: object) -> str:
    """각인 문자열의 설명어를 한 글자 토큰으로 바꾸고 공백을 지운 뒤 대문자로 맞춘다."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    for word, token in OCR_TOKENS.items():
        text = text.replace(word, token)
    return "".join(text.split()).upper()


def visible_texts(boxes: pd.DataFrame) -> pd.Series:
    """bbox마다 보이는 면(drug_dir)의 정규화 각인. 면을 모르면 NaN."""
    texts = pd.Series(np.nan, index=boxes.index, dtype=object)
    for face, column in FACE_COLUMNS.items():
        mask = boxes["drug_dir"].eq(face)
        texts[mask] = boxes.loc[mask, column].map(normalize_imprint)
    return texts


def ocr_box_table() -> pd.DataFrame:
    """대회 v1 라벨과 AI Hub 학습 라벨의 bbox마다 사진 경로·나눔·보이는 면 각인을 붙인다."""
    ann_df, _ = load_annotations(RAW_DIR / "train_annotations")
    key = ["source_json", "annotation_id"]
    meta = ann_df[[*key, *AIHUB_TEXT_COLUMNS]].assign(
        annotation_id=lambda table: pd.to_numeric(table["annotation_id"]).astype("Int64")
    )
    competition = LABELS.assign(
        annotation_id=pd.to_numeric(LABELS["annotation_id"], errors="coerce").astype("Int64")
    ).merge(meta, on=key, how="left", validate="many_to_one")
    images_by_name = {path.name: path for path in TRAIN_IMAGES}
    competition["path"] = competition["file_name"].map(images_by_name)
    competition["split"] = competition["file_name"].map(SPLIT_TABLE.set_index("file_name")["split"])
    competition["source"] = "competition"
    external = AIHUB_LABELS.reindex(
        columns=["file_name", "category_id", *AIHUB_BOX_NUMBERS[:4], *AIHUB_TEXT_COLUMNS]
    ).copy()
    external["path"] = [
        AIHUB_YOLO_DIR / "images" / "train" / name for name in external["file_name"]
    ]
    external["split"] = "train"
    external["source"] = "aihub"
    columns = ["file_name", "path", "split", "source", "category_id", "x", "y", "w", "h"]
    boxes = pd.concat(
        [competition[columns + AIHUB_TEXT_COLUMNS], external[columns + AIHUB_TEXT_COLUMNS]],
        ignore_index=True,
    )
    boxes["category_id"] = boxes["category_id"].astype(int)
    boxes["text"] = visible_texts(boxes)
    return boxes


def imprint_dictionary(boxes: pd.DataFrame, class_ids: list[int]) -> pd.DataFrame:
    """모델 클래스마다 앞·뒷면 정규화 각인 표. 약의 각인이 JSON마다 다르면 가장 많은 쪽을 쓴다."""
    known = boxes.dropna(subset=["drug_dir"]).copy()
    for column in FACE_COLUMNS.values():
        known[column] = known[column].map(normalize_imprint)
    counts = (
        known.groupby(["category_id", "print_front", "print_back"]).size().rename("n").reset_index()
    )
    counts = counts.sort_values(["category_id", "n"], ascending=[True, False])
    conflicts = sorted(
        int(value) for value in counts.loc[counts["category_id"].duplicated(), "category_id"]
    )
    if conflicts:
        print(f"JSON마다 각인이 다른 클래스 {len(conflicts)}종(가장 많은 각인 사용): {conflicts}")
    table = counts.drop_duplicates("category_id").set_index("category_id")
    if missing := sorted(set(class_ids) - set(table.index)):
        raise ValueError(f"각인 정보가 없는 모델 클래스가 있습니다: {missing}")
    return table.loc[class_ids, ["print_front", "print_back"]]


def ctc_length(text: str) -> int:
    """CTC가 text를 내는 데 필요한 최소 출력 길이(같은 글자가 이어지면 사이에 blank가 필요)."""
    return len(text) + sum(left == right for left, right in zip(text, text[1:], strict=False))


def square_crop(image: np.ndarray, box: np.ndarray, size: int) -> np.ndarray:
    """bbox(x, y, w, h) 중심에서 긴 변 × OCR_CROP_MARGIN 정사각형을 size×size로 자른다."""
    x, y, w, h = box
    scale = size / (max(w, h) * OCR_CROP_MARGIN)
    matrix = np.array(
        [
            [scale, 0, size / 2 - (x + w / 2) * scale],
            [0, scale, size / 2 - (y + h / 2) * scale],
        ]
    )
    return cv2.warpAffine(image, matrix, (size, size), borderMode=cv2.BORDER_REPLICATE)


def load_crops(boxes: pd.DataFrame) -> np.ndarray:
    """bbox마다 square_crop을 읽어 boxes 순서대로 (N, S, S, 3) uint8로 반환한다."""
    size = round(OCR_IMGSZ * OCR_CROP_MARGIN)

    def read(item: tuple[str, pd.DataFrame]) -> list[tuple[int, np.ndarray]]:
        path, rows = item
        image = cv2.imread(str(path))
        if image is None:
            raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {path}")
        return [
            (index, square_crop(image, box, size))
            for index, box in zip(
                rows.index, rows[["x", "y", "w", "h"]].to_numpy(float), strict=True
            )
        ]

    boxes = boxes.reset_index(drop=True)
    groups = list(boxes.groupby(boxes["path"].astype(str), sort=False))
    with ThreadPool(NUM_THREADS) as pool:
        results = [pair for pairs in pool.map(read, groups) for pair in pairs]
    crops = np.empty((len(boxes), size, size, 3), dtype=np.uint8)
    for index, crop in results:
        crops[index] = crop
    return crops


def crop_view(
    crop: np.ndarray, angle: float, scale: float = 1.0, shift: tuple[float, float] = (0.0, 0.0)
) -> np.ndarray:
    """crop을 가운데 기준으로 돌리고 키운 뒤 가운데 OCR_IMGSZ 영역을 자른다."""
    size = crop.shape[0]
    matrix = cv2.getRotationMatrix2D((size / 2, size / 2), angle, scale)
    matrix[:, 2] += np.asarray(shift) - (size - OCR_IMGSZ) / 2
    return cv2.warpAffine(crop, matrix, (OCR_IMGSZ, OCR_IMGSZ), borderMode=cv2.BORDER_REPLICATE)


def augment_crop(crop: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """학습 crop을 임의 각도로 돌리고 bbox 오차만큼 크기·위치를 흔든 뒤 밝기·대비를 바꾼다."""
    view = crop_view(
        crop,
        angle=rng.uniform(0, 360),
        scale=rng.uniform(0.9, 1.1),
        shift=tuple(rng.uniform(-0.05, 0.05, 2) * OCR_IMGSZ),
    )
    return cv2.convertScaleAbs(view, alpha=rng.uniform(0.8, 1.2), beta=rng.uniform(-20, 20))


def to_tensor(images: np.ndarray) -> torch.Tensor:
    """(N, H, W, 3) uint8 BGR을 (N, 3, H, W) 0~1 텐서로 바꿔 장치에 올린다."""
    return torch.from_numpy(images).permute(0, 3, 1, 2).float().div(255).to(DEVICE)


class ImprintOCR(torch.nn.Module):
    """알약 crop에서 각인 문자열을 읽는 CRNN(VGG형 CNN + BiLSTM). CTC로 학습한다."""

    def __init__(self, n_symbols: int, hidden: int = 256) -> None:
        super().__init__()

        def conv(in_channels: int, out_channels: int) -> list[torch.nn.Module]:
            return [
                torch.nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                torch.nn.BatchNorm2d(out_channels),
                torch.nn.ReLU(inplace=True),
            ]

        # 128×128 입력 → 세로 8 × 가로 32. 가로를 덜 줄여 글자 순서를 남긴다.
        self.features = torch.nn.Sequential(
            *conv(3, 64),
            torch.nn.MaxPool2d(2),
            *conv(64, 128),
            torch.nn.MaxPool2d(2),
            *conv(128, 256),
            *conv(256, 256),
            torch.nn.MaxPool2d((2, 1)),
            *conv(256, 512),
            *conv(512, 512),
            torch.nn.MaxPool2d((2, 1)),
        )
        self.rnn = torch.nn.LSTM(
            512, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1
        )
        self.classifier = torch.nn.Linear(2 * hidden, n_symbols + 1)  # 0번은 CTC blank

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """(B, 3, H, W) → 글자별 로그 확률 (B, T, 글자 수 + 1)."""
        features = self.features(images).mean(dim=2)  # 세로 평균 → (B, C, T)
        sequence, _ = self.rnn(features.transpose(1, 2))
        return self.classifier(sequence).log_softmax(-1)


class OcrCandidates:
    """모델 클래스마다 앞·뒷면 각인을 글자 번호로 바꾼 후보 표."""

    def __init__(self, dictionary: pd.DataFrame, vocab: list[str]) -> None:
        self.class_ids = [int(value) for value in dictionary.index]
        self.texts = sorted({text for text in dictionary.to_numpy().ravel()})
        self.vocab = vocab
        self.char_index = {char: index + 1 for index, char in enumerate(vocab)}
        text_index = {text: index for index, text in enumerate(self.texts)}
        # 클래스 × 면 → 후보 텍스트 번호
        self.class_texts = np.array(
            [[text_index[text] for text in row] for row in dictionary.to_numpy()]
        )
        too_long = [text for text in self.texts if ctc_length(text) > OCR_SEQ_LEN]
        if too_long:
            raise ValueError(f"OCR 출력 길이 {OCR_SEQ_LEN}보다 긴 각인이 있습니다: {too_long}")

    def encode(self, text: str) -> list[int]:
        """각인 문자열을 글자 번호 목록으로 바꾼다."""
        return [self.char_index[char] for char in text]


@torch.inference_mode()
def ocr_class_log_probs(
    model: ImprintOCR, crops: np.ndarray, candidates: OcrCandidates
) -> np.ndarray:
    """crop마다 모델 클래스의 각인이 나올 CTC 로그 확률 (N, 클래스 수).

    crop을 OCR_ROTATIONS 방향으로 돌려 가장 잘 읽히는 방향을 쓰고, 앞·뒷면 중 더 맞는 면을 쓴다.
    """
    model.eval()
    angles = [360 * index / OCR_ROTATIONS for index in range(OCR_ROTATIONS)]
    targets = [torch.tensor(candidates.encode(text), dtype=torch.long) for text in candidates.texts]
    text_scores = []
    for start in range(0, len(crops), OCR_SCORE_BATCH):
        views = np.stack(
            [
                crop_view(crop, angle)
                for crop in crops[start : start + OCR_SCORE_BATCH]
                for angle in angles
            ]
        )
        # CTC는 MPS에 없어 CPU에서 계산한다.
        log_probs = model(to_tensor(views)).float().cpu().transpose(0, 1)  # (T, B, C)
        n_views = log_probs.shape[1]
        scores = torch.empty(n_views, len(targets))
        for index, target in enumerate(targets):
            scores[:, index] = -torch.nn.functional.ctc_loss(
                log_probs,
                target.repeat(n_views),
                input_lengths=torch.full((n_views,), log_probs.shape[0], dtype=torch.long),
                target_lengths=torch.full((n_views,), len(target), dtype=torch.long),
                reduction="none",
            )
        text_scores.append(scores.view(-1, len(angles), len(targets)).amax(dim=1))
    text_scores = torch.cat(text_scores).numpy()  # (N, 후보 텍스트 수)
    return text_scores[:, candidates.class_texts].max(axis=2)  # 면 중 큰 값 → (N, 클래스 수)


def ocr_probabilities(class_log_probs: np.ndarray) -> np.ndarray:
    """클래스 로그 확률을 OCR_TEMPERATURE로 나눠 softmax한 OCR 확률."""
    logits = class_log_probs / OCR_TEMPERATURE
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    return probs / probs.sum(axis=1, keepdims=True)


def ocr_accuracy(
    model: ImprintOCR, crops: np.ndarray, categories: np.ndarray, candidates: OcrCandidates
) -> float:
    """정답 crop에서 OCR 확률이 가장 큰 클래스가 정답 클래스와 같은 비율."""
    scores = ocr_class_log_probs(model, crops, candidates)
    predicted = np.asarray(candidates.class_ids)[scores.argmax(axis=1)]
    return float((predicted == categories).mean())


def train_ocr(boxes: pd.DataFrame, candidates: OcrCandidates) -> Path:
    """학습 bbox crop으로 CRNN을 CTC로 학습하고 검증 정확도가 가장 높은 가중치를 반환한다."""
    config = {key: EXPERIMENT_CONFIG[key] for key in OCR_TRAINING_KEYS}
    complete_path = OCR_DIR / "completed.json"
    best_path = OCR_DIR / "best.pt"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text(encoding="utf-8"))
        if complete["config"] == json.loads(json.dumps(config)) and best_path.is_file():
            print(f"완료된 OCR 학습 사용: {best_path}")
            return best_path
        print("저장된 OCR 기록과 설정이 달라 OCR을 다시 학습합니다.")
    OCR_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)

    train = boxes[boxes["split"].eq("train") & boxes["text"].notna()].copy()
    train = train[train["text"].map(ctc_length).le(OCR_SEQ_LEN)]
    if TRAIN_FRACTION < 1:
        train = train.sample(frac=TRAIN_FRACTION, random_state=SEED)
    val = boxes[boxes["split"].eq("val") & boxes["category_id"].isin(candidates.class_ids)]
    print(f"OCR crop 읽는 중: train {len(train)}개, validation {len(val)}개")
    train_crops, val_crops = load_crops(train), load_crops(val)
    train_targets = [candidates.encode(text) for text in train["text"]]
    val_categories = val["category_id"].to_numpy()

    model = ImprintOCR(len(candidates.vocab)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=OCR_LR, weight_decay=1e-4)
    steps_per_epoch = math.ceil(len(train) / OCR_BATCH)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=OCR_LR, total_steps=TRAIN_OCR_EPOCHS * steps_per_epoch, pct_start=0.1
    )
    history, best_accuracy, best_epoch = [], -1.0, 0
    for epoch in range(1, TRAIN_OCR_EPOCHS + 1):
        model.train()
        started, losses = perf_counter(), []
        order = rng.permutation(len(train))
        for start in range(0, len(order), OCR_BATCH):
            batch = order[start : start + OCR_BATCH]
            images = to_tensor(np.stack([augment_crop(train_crops[i], rng) for i in batch]))
            targets = [train_targets[i] for i in batch]
            log_probs = model(images).float().cpu().transpose(0, 1)  # (T, B, C)
            loss = torch.nn.functional.ctc_loss(
                log_probs,
                torch.tensor([char for target in targets for char in target], dtype=torch.long),
                input_lengths=torch.full((len(batch),), log_probs.shape[0], dtype=torch.long),
                target_lengths=torch.tensor([len(target) for target in targets], dtype=torch.long),
                zero_infinity=True,
            )
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss))
        accuracy = ocr_accuracy(model, val_crops, val_categories, candidates)
        history.append(
            {
                "epoch": epoch,
                "train_ctc_loss": float(np.mean(losses)),
                "val_imprint_accuracy": accuracy,
                "lr": scheduler.get_last_lr()[0],
                "epoch_seconds": perf_counter() - started,
            }
        )
        print(json.dumps(history[-1], ensure_ascii=False))
        pd.DataFrame(history).to_csv(OCR_DIR / "history.csv", index=False)
        if accuracy > best_accuracy:
            best_accuracy, best_epoch = accuracy, epoch
            torch.save(
                {"state_dict": model.state_dict(), "vocab": candidates.vocab, "epoch": epoch},
                best_path,
            )
        elif epoch - best_epoch >= OCR_PATIENCE:
            print(f"OCR early stopping: {OCR_PATIENCE}에폭 개선 없음, best epoch={best_epoch}")
            break
    write_json(
        complete_path,
        {
            "config": config,
            "best_epoch": best_epoch,
            "best_val_imprint_accuracy": best_accuracy,
            "train_crops": len(train),
            "val_crops": len(val),
            "vocab": candidates.vocab,
        },
    )
    del model
    gc.collect()
    empty_device_cache()
    return best_path


def load_ocr(path: Path, candidates: OcrCandidates) -> ImprintOCR:
    """저장한 OCR 가중치를 읽는다. 글자 목록이 지금 후보와 다르면 멈춘다."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if checkpoint["vocab"] != candidates.vocab:
        raise ValueError("저장한 OCR의 글자 목록이 지금 각인 사전과 다릅니다.")
    model = ImprintOCR(len(candidates.vocab))
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(DEVICE).eval()


def rerank_with_ocr(
    predictions: pd.DataFrame, probs: np.ndarray, candidates: OcrCandidates
) -> pd.DataFrame:
    """OCR이 다른 클래스를 강하게 가리킬 때만 클래스를 바꾼다.

    OCR 최고 클래스 확률 ≥ OCR_SWITCH_PROB이고 YOLO 클래스의 OCR 확률 ≤ OCR_KEEP_PROB일 때다.
    """
    position = {code: index for index, code in enumerate(candidates.class_ids)}
    yolo_column = predictions["category_id"].map(position).to_numpy()
    rows = np.arange(len(predictions))
    best = probs.argmax(axis=1)
    switch = (
        (best != yolo_column)
        & (probs[rows, best] >= OCR_SWITCH_PROB)
        & (probs[rows, yolo_column] <= OCR_KEEP_PROB)
    )
    result = predictions.copy()
    result["category_id_yolo"] = result["category_id"]
    result["category_id_ocr"] = np.asarray(candidates.class_ids)[best]
    result["ocr_prob_yolo"] = probs[rows, yolo_column]
    result["ocr_prob_best"] = probs[rows, best]
    result["ocr_switched"] = switch
    result.loc[switch, "category_id"] = result.loc[switch, "category_id_ocr"]
    return result


def xyxy_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """xyxy 박스 a(n개)와 b(m개)의 IoU 표(n × m)."""
    width = (
        np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0])
    ).clip(0)
    height = (
        np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1])
    ).clip(0)
    inter = width * height
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter)


def match_correct(
    pred_classes: np.ndarray, true_classes: np.ndarray, iou: np.ndarray
) -> np.ndarray:
    """ultralytics BaseValidator.match_predictions(기본 greedy)와 같은 IoU 10단계 정답 표.

    pred는 신뢰도 내림차순, iou는 (정답 수, 예측 수)이다.
    """
    correct = np.zeros((len(pred_classes), len(IOUV)), dtype=bool)
    iou = iou * (true_classes[:, None] == pred_classes[None, :])
    matched = np.zeros((iou.shape[0], len(IOUV)), dtype=bool)
    for j in np.flatnonzero((iou >= IOUV.min()).any(0)):
        available = np.where(matched, 0, iou[:, j, None])
        k = available.argmax(0)
        correct[j] = available[k, range(len(IOUV))] >= IOUV
        matched[k, range(len(IOUV))] |= correct[j]
    return correct


def holdout_map75_95(predictions: pd.DataFrame) -> float:
    """예측(file_name, xyxy, category_id, conf)과 validation 정답으로 대회 클래스 mAP75-95를 구한다.

    ultralytics 검증과 같은 매칭(match_correct)과 AP 계산(ap_per_class)을 쓴다.
    """
    val_files = sorted(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    truth = LABELS[LABELS["file_name"].isin(val_files)]
    comp_ids = set(CLASS_MAP["category_id"])
    predictions = predictions[predictions["category_id"].isin(comp_ids)]
    truth_by_file = dict(tuple(truth.groupby("file_name")))
    pred_by_file = dict(tuple(predictions.groupby("file_name")))
    tps, confs, pred_cls, target_cls = [], [], [], []
    for file_name in val_files:
        gt = truth_by_file.get(file_name, truth.iloc[:0])
        target_cls.append(gt["category_id"].to_numpy())
        pred = pred_by_file.get(file_name)
        if pred is None or pred.empty:
            continue
        pred = pred.sort_values("conf", ascending=False)
        pred_xyxy = np.array(pred["xyxy"].tolist(), dtype=float)
        gt_xyxy = gt[["x", "y", "w", "h"]].to_numpy(float)
        gt_xyxy[:, 2:] += gt_xyxy[:, :2]
        iou = xyxy_iou(gt_xyxy, pred_xyxy) if len(gt) else np.zeros((0, len(pred)))
        tps.append(match_correct(pred["category_id"].to_numpy(), gt["category_id"].to_numpy(), iou))
        confs.append(pred["conf"].to_numpy())
        pred_cls.append(pred["category_id"].to_numpy())
    if not tps:
        return 0.0
    ap = ap_per_class(
        np.concatenate(tps),
        np.concatenate(confs),
        np.concatenate(pred_cls),
        np.concatenate(target_cls),
    )[5]
    return float(ap75_95(ap).mean()) if len(ap) else 0.0


def switch_outcomes(reranked: pd.DataFrame) -> dict:
    """클래스를 바꾼 예측을 IoU 0.5 이상인 가장 가까운 정답과 비교해 바뀐 결과를 센다."""
    truth = LABELS.set_index("file_name")
    counts = {"wrong_to_right": 0, "right_to_wrong": 0, "wrong_to_wrong": 0, "no_match": 0}
    for row in reranked[reranked["ocr_switched"]].itertuples():
        gt = truth.loc[[row.file_name]]
        gt_xyxy = gt[["x", "y", "w", "h"]].to_numpy(float)
        gt_xyxy[:, 2:] += gt_xyxy[:, :2]
        iou = xyxy_iou(np.array([row.xyxy], dtype=float), gt_xyxy)[0]
        if iou.max() < 0.5:
            counts["no_match"] += 1
            continue
        target = int(gt["category_id"].iloc[iou.argmax()])
        if row.category_id == target:
            counts["wrong_to_right"] += 1
        elif row.category_id_yolo == target:
            counts["right_to_wrong"] += 1
        else:
            counts["wrong_to_wrong"] += 1
    return counts


def plot_ocr_history(output_path: Path) -> None:
    """OCR 에폭별 학습 CTC loss와 검증 각인 분류 정확도를 그린다."""
    history = pd.read_csv(OCR_DIR / "history.csv")
    best = history.loc[history["val_imprint_accuracy"].idxmax()]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_ctc_loss"], marker=".")
    axes[0].set(title="OCR train CTC loss", xlabel="epoch")
    axes[1].plot(history["epoch"], history["val_imprint_accuracy"], marker=".")
    axes[1].scatter(
        best["epoch"], best["val_imprint_accuracy"], marker="*", s=300, color="red", zorder=3
    )
    axes[1].set(title="OCR validation imprint accuracy", xlabel="epoch", ylim=(0, 1.05))
    for axis in axes:
        axis.axvline(best["epoch"], color="gray", linestyle=":")
        axis.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
if not USE_OCR:
    print("USE_OCR=False: 각인 OCR을 건너뜁니다.")
else:
    train_names = yaml.safe_load(TRAIN_DATA_YAML.read_text(encoding="utf-8"))["names"]
    # 모델 클래스 순서(yolo_idx)의 category_id. 외부 전용 클래스 이름은 약 코드 숫자다.
    comp_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    MODEL_CLASS_IDS = [
        int(comp_by_index.get(index, name))
        for index, name in sorted((int(key), value) for key, value in train_names.items())
    ]
    OCR_BOXES = ocr_box_table()
    OCR_DICTIONARY = imprint_dictionary(OCR_BOXES, MODEL_CLASS_IDS)
    OCR_VOCAB = sorted(
        set("".join(OCR_DICTIONARY.to_numpy().ravel())) | set("".join(OCR_BOXES["text"].dropna()))
    )
    OCR_CANDIDATES = OcrCandidates(OCR_DICTIONARY, OCR_VOCAB)
    text_known = OCR_BOXES["text"].notna()
    display(
        pd.DataFrame(
            {
                "bbox": OCR_BOXES.groupby(["source", "split"]).size(),
                "보이는 면 각인 있음": OCR_BOXES[text_known].groupby(["source", "split"]).size(),
                "빈 각인(글자 없는 면)": OCR_BOXES[text_known & OCR_BOXES["text"].eq("")]
                .groupby(["source", "split"])
                .size(),
            }
        )
        .fillna(0)
        .astype(int)
    )
    print(
        f"모델 클래스 {len(MODEL_CLASS_IDS)}종, 서로 다른 각인 후보 {len(OCR_CANDIDATES.texts)}개, "
        f"글자 {len(OCR_VOCAB)}종: {''.join(OCR_VOCAB)}"
    )
    shared = OCR_DICTIONARY.reset_index().melt("category_id", value_name="text")
    shared = (
        shared[shared["text"].ne("")]
        .groupby("text")["category_id"]
        .agg(lambda ids: sorted(set(ids)))
    )
    shared = shared[shared.map(len) > 1]
    if len(shared):
        print(f"여러 클래스가 같은 각인을 쓰는 경우 {len(shared)}개 (OCR로 구별 불가):")
        display(shared.to_frame("category_id"))

    OCR_WEIGHTS = train_ocr(OCR_BOXES, OCR_CANDIDATES)
    OCR_DICTIONARY.to_csv(OCR_DIR / "imprint_dictionary.csv")
    plot_ocr_history(OCR_DIR / "ocr_history.png")
    OCR_MODEL = load_ocr(OCR_WEIGHTS, OCR_CANDIDATES)

    # 5절에서 저장한 conf=0.001 검증 예측에 OCR 재분류를 적용하고 mAP75-95를 비교한다.
    records = json.loads(
        (EXPERIMENT_DIR / "evaluation" / f"predictions_conf_{EVAL_CONF:g}.json").read_text(
            encoding="utf-8"
        )
    )
    category_by_index = dict(enumerate(MODEL_CLASS_IDS))
    val_predictions = pd.DataFrame(
        [
            {
                "file_name": record["file_name"],
                "xyxy": xyxy,
                "category_id": category_by_index[index],
                "conf": conf,
            }
            for record in records
            for xyxy, index, conf in zip(
                record["xyxy"], record["yolo_idx"], record["conf"], strict=True
            )
        ]
    )
    images_by_name = {path.name: path for path in TRAIN_IMAGES}
    pred_xyxy = pd.DataFrame(val_predictions["xyxy"].tolist(), columns=["x1", "y1", "x2", "y2"])
    crop_boxes = pd.DataFrame(
        {
            "path": val_predictions["file_name"].map(images_by_name),
            "x": pred_xyxy["x1"],
            "y": pred_xyxy["y1"],
            "w": pred_xyxy["x2"] - pred_xyxy["x1"],
            "h": pred_xyxy["y2"] - pred_xyxy["y1"],
        }
    )
    started = perf_counter()
    val_probs = ocr_probabilities(
        ocr_class_log_probs(OCR_MODEL, load_crops(crop_boxes), OCR_CANDIDATES)
    )
    ocr_seconds = perf_counter() - started
    VAL_RERANKED = rerank_with_ocr(val_predictions, val_probs, OCR_CANDIDATES)
    VAL_RERANKED.assign(xyxy=VAL_RERANKED["xyxy"].map(json.dumps)).to_csv(
        OCR_DIR / "val_reranked_conf0.001.csv", index=False
    )
    yolo_map = holdout_map75_95(val_predictions)
    ocr_map = holdout_map75_95(VAL_RERANKED)
    gt_val = OCR_BOXES[OCR_BOXES["split"].eq("val")]
    OCR_RESULTS = {
        "val_imprint_accuracy_gt_crops": ocr_accuracy(
            OCR_MODEL, load_crops(gt_val), gt_val["category_id"].to_numpy(), OCR_CANDIDATES
        ),
        f"{MAIN_METRIC}_section5": float(PRIMARY_RESULTS[MAIN_METRIC].iloc[0]),
        f"{MAIN_METRIC}_yolo_recomputed": yolo_map,
        f"{MAIN_METRIC}_yolo_ocr": ocr_map,
        "delta": ocr_map - yolo_map,
        "n_predictions": len(VAL_RERANKED),
        "n_switched": int(VAL_RERANKED["ocr_switched"].sum()),
        **switch_outcomes(VAL_RERANKED),
        "ocr_ms_per_image": ocr_seconds * 1000 / VAL_RERANKED["file_name"].nunique(),
        "ocr_rotations": OCR_ROTATIONS,
        "ocr_temperature": OCR_TEMPERATURE,
        "ocr_switch_prob": OCR_SWITCH_PROB,
        "ocr_keep_prob": OCR_KEEP_PROB,
    }
    write_json(OCR_DIR / "ocr_results.json", OCR_RESULTS)
    display(pd.Series(OCR_RESULTS, name="OCR 재분류 (validation, conf=0.001)").to_frame())
    print("5절 값과 다시 계산한 YOLO 값이 거의 같아야 OCR 전후 비교를 믿을 수 있습니다.")
    switched = VAL_RERANKED[VAL_RERANKED["ocr_switched"]]
    if len(switched):
        display(
            switched[
                [
                    "file_name",
                    "conf",
                    "category_id_yolo",
                    "category_id",
                    "ocr_prob_yolo",
                    "ocr_prob_best",
                ]
            ]
        )

## 6. Validation 정답 bbox · 예측 bbox 이미지

conf=0.25 평가에서 실제 사용한 예측을 가져옵니다. 평가를 다시 추론하지 않습니다.
**왼쪽: 정답(초록), 가운데: 예측(주황·점수 표시), 오른쪽: 겹쳐 보기**입니다.
정답은 보정된 v1 라벨, 예측은 원본 이미지 좌표입니다.

고정한 모든 validation 이미지 비교본을 저장합니다. 아래 조회 셀에서 시작 위치를
바꿔 원하는 페이지를 볼 수 있습니다. 표시 신뢰도는 평가 conf=0.001과 별도인 VIS_CONF=0.25입니다.

In [ ]:
def draw_box(axis: object, xyxy: list[float], label: str, color: str, linestyle: str = "-") -> None:
    """원본 픽셀 좌표로 박스와 클래스·점수를 그린다."""
    x1, y1, x2, y2 = xyxy
    axis.add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor=color,
            linewidth=1.8,
            linestyle=linestyle,
        )
    )
    axis.text(
        x1,
        max(0, y1 - 5),
        label,
        color="white",
        fontsize=8,
        bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 2},
    )


def save_validation_images() -> list[Path]:
    """고정 validation의 모든 이미지에 정답·실제 평가 예측 박스를 그려 저장한다."""
    eval_dir = EXPERIMENT_DIR / "evaluation"
    records = json.loads(
        (eval_dir / f"predictions_conf_{VIS_CONF:g}.json").read_text(encoding="utf-8")
    )
    expected_files = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    if {record["file_name"] for record in records} != expected_files:
        raise AssertionError("시각화 대상이 validation 목록과 다릅니다.")
    images_by_name = {path.name: path for path in TRAIN_IMAGES}
    category_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    image_dir = EXPERIMENT_DIR / "val_images"
    image_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for record in sorted(records, key=lambda item: item["file_name"]):
        file_name = record["file_name"]
        bgr = cv2.imread(str(images_by_name[file_name]))
        if bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {file_name}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 3, figsize=(15, 6))
        for axis, title in zip(
            axes, ("Ground truth", f"Prediction conf>{VIS_CONF:g}", "Overlay"), strict=True
        ):
            axis.imshow(rgb)
            axis.set_title(title)
            axis.axis("off")
        truth = LABELS.loc[LABELS["file_name"] == file_name]
        for box in truth.itertuples():
            coords = [box.x, box.y, box.x + box.w, box.y + box.h]
            for index in (0, 2):
                draw_box(axes[index], coords, f"GT {box.category_id}", "green")
        for coords, class_index, score in zip(
            record["xyxy"], record["yolo_idx"], record["conf"], strict=True
        ):
            category = category_by_index[class_index]
            for index in (1, 2):
                draw_box(axes[index], coords, f"Pred {category} {score:.3f}", "darkorange", "--")
        if not record["xyxy"]:
            axes[1].text(0.5, 0.05, "No predictions", transform=axes[1].transAxes, ha="center")
        fig.suptitle(f"Holdout | {file_name}", fontsize=9)
        fig.tight_layout()
        path = image_dir / f"{Path(file_name).stem}.jpg"
        fig.savefig(path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        paths.append(path)
    return paths


def show_validation_images(start: int = 0, count: int = 4) -> None:
    """저장된 검증 비교 이미지를 지정한 페이지부터 노트북에 표시한다."""
    paths = sorted((EXPERIMENT_DIR / "val_images").glob("*.jpg"))
    if not paths:
        raise FileNotFoundError("먼저 검증 비교 이미지 저장 셀을 실행하세요.")
    if start < 0 or count < 1 or start >= len(paths):
        raise ValueError(f"start는 0~{len(paths) - 1}, count는 1 이상이어야 합니다.")
    print(f"Holdout: {start + 1}~{min(start + count, len(paths))} / 전체 {len(paths)}장")
    for path in paths[start : start + count]:
        display(Image(filename=str(path)))


paths = save_validation_images()
print(f"GT/예측 비교 이미지 {len(paths)}장 저장")

In [ ]:
# start=0, 4, 8, ...로 바꿔 모든 검증 이미지를 볼 수 있다.
show_validation_images(start=0, count=4)

## 7. 추론속도 · 이미지별 꺾은선 그래프 · 단일 holdout 결과

단일 학습의 best.pt와 validation 이미지로 측정합니다. **batch=1, FP32, 640×640,
conf=0.001, NMS 없음(one2one 헤드), max_det=4**를 적용합니다.
warmup 5회 후 validation 전체를 3회 반복하며 매 측정 전후 장치(CUDA·MPS)를 동기화합니다.

forward_ms는 모델 연산 시간입니다. pipeline_ms는 전처리·GPU 전송·forward·후처리·
Python 호출까지의 시간이며 디스크 읽기는 제외합니다.
학습 배치 16, 학습 중 검증 최대 32, 최종 평가 배치 16과 구분합니다.

각 이미지의 반복 측정 평균으로 forward·전체 추론시간을 꺾은선 그래프로 표시합니다.
각 평균은 같은 이미지의 반복 측정값으로 계산합니다.

In [ ]:
def plot_inference_timings(timings: pd.DataFrame, output_path: Path) -> None:
    """검증 이미지별 반복 측정 평균 추론시간을 꺾은선 그래프로 저장하고 표시한다."""
    required = {"file_name", "forward_ms", "pipeline_ms"}
    if timings.empty or not required <= set(timings.columns):
        raise ValueError("이미지별 추론시간 기록이 없습니다.")
    image_order = timings["file_name"].drop_duplicates().tolist()
    means = timings.groupby("file_name")[["forward_ms", "pipeline_ms"]].mean().reindex(image_order)
    positions = np.arange(1, len(means) + 1)
    fig, axis = plt.subplots(figsize=(12, 5))
    axis.plot(positions, means["forward_ms"], marker=".", label="Model forward")
    axis.plot(positions, means["pipeline_ms"], marker=".", label="Full inference pipeline")
    axis.set(
        title="Holdout | Mean inference time per validation image",
        xlabel="Validation image index",
        ylabel="Time (ms)",
    )
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()
    plt.close(fig)


def benchmark_baseline(weights: Path) -> dict:
    """warmup·GPU 동기화 후 batch=1의 모델/전체 추론 시간을 기록한다."""
    val_names = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    loaded = []
    for path in sorted(TRAIN_IMAGES):
        if path.name in val_names:
            image = cv2.imread(str(path))
            if image is None:
                raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
            loaded.append((path.name, image))
    if not loaded or BENCHMARK_WARMUP < 1 or BENCHMARK_ROUNDS < 1:
        raise ValueError("검증 이미지, warmup, 반복 수는 1 이상이어야 합니다.")
    model = YOLO(str(weights))
    model.to(DEVICE)
    model.model.eval()
    predict_args = {
        "imgsz": IMGSZ,
        "batch": 1,
        "rect": False,
        "device": DEVICE,
        "quantize": 32,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "max_det": MAX_DET,
        "augment": False,
        "save": False,
        "verbose": False,
        "stream": False,
    }
    rows = []
    try:
        with torch.inference_mode():
            for _ in range(BENCHMARK_WARMUP):
                model.predict(source=loaded[0][1], **predict_args)
            sync_device()
            for repetition in range(BENCHMARK_ROUNDS):
                for file_name, image in loaded:
                    sync_device()
                    start = perf_counter()
                    prediction = model.predict(source=image, **predict_args)[0]
                    sync_device()
                    elapsed = (perf_counter() - start) * 1000
                    if len(prediction.boxes) > MAX_DET:
                        raise AssertionError("속도 측정 예측의 max_det가 다릅니다.")
                    rows.append(
                        {
                            "evaluation": "holdout",
                            "round": repetition + 1,
                            "file_name": file_name,
                            "pipeline_ms": elapsed,
                            "forward_ms": float(prediction.speed["inference"]),
                            "preprocess_ms": float(prediction.speed["preprocess"]),
                            "postprocess_ms": float(prediction.speed["postprocess"]),
                        }
                    )
    finally:
        del model
        gc.collect()
        empty_device_cache()
    timings = pd.DataFrame(rows)
    summary = {
        "evaluation": "holdout",
        "n_images": len(loaded),
        "n_measurements": len(timings),
        "forward_mean_ms": float(timings["forward_ms"].mean()),
        "pipeline_mean_ms": float(timings["pipeline_ms"].mean()),
        "pipeline_median_ms": float(timings["pipeline_ms"].median()),
        "pipeline_p95_ms": float(timings["pipeline_ms"].quantile(0.95)),
        "forward_fps": 1000 / float(timings["forward_ms"].mean()),
        "pipeline_fps": 1000 / float(timings["pipeline_ms"].mean()),
    }
    out_dir = EXPERIMENT_DIR / "evaluation"
    timings.to_csv(out_dir / "inference_timings.csv", index=False)
    plot_inference_timings(timings, out_dir / "inference_time_curves.png")
    write_json(
        out_dir / "inference_speed.json",
        {
            **summary,
            **ENVIRONMENT,
            "batch": 1,
            "imgsz": IMGSZ,
            "precision": "FP32",
            "conf": EVAL_CONF,
            "nms_iou": NMS_IOU,
            "max_det": MAX_DET,
            "nms": NMS_ENABLED,
            "agnostic_nms": False,
            "disk_io_included": False,
            "warmup": BENCHMARK_WARMUP,
            "rounds": BENCHMARK_ROUNDS,
        },
    )
    return summary


SPEED_RESULTS = pd.DataFrame([benchmark_baseline(BEST_WEIGHTS)])
SPEED_RESULTS.to_csv(EXPERIMENT_DIR / "inference_speed_holdout.csv", index=False)
FINAL_RESULTS = PRIMARY_RESULTS.merge(SPEED_RESULTS, on="evaluation", validate="1:1")
FINAL_RESULTS.to_csv(EXPERIMENT_DIR / "results_holdout.csv", index=False)
speed_columns = ["forward_mean_ms", "pipeline_mean_ms", "forward_fps", "pipeline_fps"]
FINAL_SUMMARY = FINAL_RESULTS.set_index("evaluation")[metric_columns + speed_columns].T
FINAL_SUMMARY.to_csv(EXPERIMENT_DIR / "summary_holdout.csv")
display(FINAL_RESULTS)
display(FINAL_SUMMARY)
print(f"전체 결과: {EXPERIMENT_DIR}")
print("가중치: train/weights/best.pt")
print("혼동행렬: evaluation/conf_0.001 및 conf_0.25")
print("검증 정답/예측 이미지: val_images/")

## 8. 대회 제출 csv (test 842장)

best.pt로 test 이미지 전체를 예측해 대회 제출 형식 csv를 만듭니다. test는 학습·검증에 쓰지 않았습니다.
검증 점수와 같은 조건이 되도록 **conf=0.001, 위에서 정한 후처리(`NMS_ENABLED`, NMS IoU=0.5),
max_det=4, FP32**로 예측합니다. mAP 채점이라 점수 낮은 박스를 남겨도 점수가 떨어지지 않으므로
conf는 평가와 같은 0.001을 씁니다. 외부 전용 클래스 예측은 `SUBMIT_EXTRA_CLASSES`가 True면
약 코드(모델 클래스 이름)를 category_id로 넣고, False면 지웁니다. test에는 대회 56종 밖의 알약도 있습니다.

| 열 | 값 |
|---|---|
| annotation_id | 1부터 매긴 행 번호 (image_id 순, 같은 사진 안에서는 score 높은 순) |
| image_id | test 파일 이름의 숫자 (12.png → 12) |
| category_id | 원본 알약 클래스 번호 (YOLO 번호가 아님) |
| bbox_x, bbox_y, bbox_w, bbox_h | 원본 이미지 픽셀 기준 COCO 형식(왼쪽 위 x, y, 너비, 높이) |
| score | 예측 신뢰도 |

OCR 재분류를 적용한 제출 csv는 8-2절에서 따로 만듭니다.
결과는 실험 폴더의 `submission/submission_<RUN_TAG>_conf0.001.csv`(외부 클래스 포함이면
`_extra_conf0.001.csv`)로 저장합니다.
RUN_TAG(모델·후처리·옵티마이저·lr·epoch·patience·batch·imgsz·증강·모자이크·copy-paste·max_det)는 설정 셀에서 만들며
결과 폴더 이름에도 들어갑니다.


In [ ]:
SUBMISSION_COLS = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]


def make_test_submission(weights: Path) -> tuple[pd.DataFrame, Path]:
    """test 이미지를 검증과 같은 후처리로 예측해 대회 제출 csv를 저장한다."""
    category_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    model = YOLO(str(weights))
    # 외부 전용 클래스 이름은 약 코드 숫자 문자열이다(데이터 준비의 names).
    category_by_index |= {i: int(name) for i, name in model.names.items() if i >= N_COMP_CLASSES}
    rows = []
    n_extra = 0  # 외부 전용 클래스 예측 수(SUBMIT_EXTRA_CLASSES가 False면 지운 수)
    try:
        # 경로 목록을 한 번에 넘기면 한 배치로 묶여 메모리를 넘으므로 한 장씩 예측한다.
        for path in TEST_IMAGES:
            result = model.predict(
                source=str(path),
                imgsz=IMGSZ,
                device=DEVICE,
                quantize=32,
                conf=EVAL_CONF,
                iou=NMS_IOU,
                nms=NMS_ENABLED,
                agnostic_nms=False,
                max_det=MAX_DET,
                augment=False,
                verbose=False,
            )[0]
            boxes = result.boxes
            if len(boxes) > MAX_DET:
                raise AssertionError("제출 예측의 max_det가 다릅니다.")
            image_id = int(Path(result.path).stem)  # 12.png → 12
            for (x1, y1, x2, y2), class_index, score in zip(
                boxes.xyxy.tolist(), boxes.cls.int().tolist(), boxes.conf.tolist(), strict=True
            ):
                if class_index >= N_COMP_CLASSES:
                    n_extra += 1
                    if not SUBMIT_EXTRA_CLASSES:
                        continue
                rows.append(
                    {
                        "image_id": image_id,
                        "category_id": category_by_index[class_index],
                        "bbox_x": x1,
                        "bbox_y": y1,
                        "bbox_w": x2 - x1,
                        "bbox_h": y2 - y1,
                        "score": score,
                    }
                )
    finally:
        del model
        gc.collect()
        empty_device_cache()
    # image_id 순, 같은 사진 안에서는 score 높은 순으로 정렬하고 annotation_id를 1부터 매긴다.
    submission = pd.DataFrame(rows, columns=SUBMISSION_COLS[1:]).sort_values(
        ["image_id", "score"], ascending=[True, False], ignore_index=True
    )
    submission.insert(0, "annotation_id", submission.index + 1)
    expected_ids = {int(path.stem) for path in TEST_IMAGES}
    missing_ids = expected_ids - set(submission["image_id"])
    if not set(submission["image_id"]) <= expected_ids:
        raise AssertionError("test에 없는 image_id가 있습니다.")
    if submission.isna().any().any() or (submission[["bbox_w", "bbox_h"]] <= 0).any().any():
        raise AssertionError("제출 표에 빈 값이나 크기 0 이하의 박스가 있습니다.")
    if n_extra:
        action = "넣었습니다" if SUBMIT_EXTRA_CLASSES else "뺐습니다"
        print(f"외부 전용 클래스 예측 {n_extra}개를 제출에 {action}.")
    if missing_ids:
        print(f"예측이 하나도 없는 test 이미지 {len(missing_ids)}장: {sorted(missing_ids)[:10]}")
    extra_tag = "_extra" if SUBMIT_EXTRA_CLASSES else ""
    out_path = (
        EXPERIMENT_DIR / "submission" / f"submission_{RUN_TAG}{extra_tag}_conf{EVAL_CONF:g}.csv"
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(out_path, index=False, encoding="utf-8")
    return submission, out_path


SUBMISSION, SUBMISSION_PATH = make_test_submission(BEST_WEIGHTS)
display(SUBMISSION.head(10))
per_image = SUBMISSION.groupby("image_id").size()
print(
    f"test {SUBMISSION['image_id'].nunique()}/{len(TEST_IMAGES)}장, 박스 {len(SUBMISSION)}개 "
    f"(사진당 평균 {per_image.mean():.2f}개, score≥{VIS_CONF:g} 박스 "
    f"{int((SUBMISSION['score'] >= VIS_CONF).sum())}개)"
)
print(f"제출 파일: {SUBMISSION_PATH}")

### 8-2. OCR 재분류 제출 csv

위 제출 csv의 박스마다 5-2절 OCR 재분류를 적용한 csv를 따로 만듭니다. 박스·점수는 그대로이고
클래스만 바뀝니다. `SUBMIT_EXTRA_CLASSES`가 False면 외부 전용 클래스로 바뀐 박스는 지웁니다.
파일 이름은 `submission_<RUN_TAG>..._ocr_conf0.001.csv`입니다. 어느 쪽을 낼지는 5-2절 validation 비교를 보고 정합니다.


In [ ]:
if USE_OCR:
    test_by_id = {int(path.stem): path for path in TEST_IMAGES}
    test_boxes = pd.DataFrame(
        {
            "path": SUBMISSION["image_id"].map(test_by_id),
            "x": SUBMISSION["bbox_x"],
            "y": SUBMISSION["bbox_y"],
            "w": SUBMISSION["bbox_w"],
            "h": SUBMISSION["bbox_h"],
        }
    )
    test_probs = ocr_probabilities(
        ocr_class_log_probs(OCR_MODEL, load_crops(test_boxes), OCR_CANDIDATES)
    )
    test_reranked = rerank_with_ocr(SUBMISSION, test_probs, OCR_CANDIDATES)
    if not SUBMIT_EXTRA_CLASSES:
        test_reranked = test_reranked[
            test_reranked["category_id"].isin(set(CLASS_MAP["category_id"]))
        ]
    OCR_SUBMISSION = test_reranked[SUBMISSION_COLS[1:]].reset_index(drop=True)
    OCR_SUBMISSION.insert(0, "annotation_id", OCR_SUBMISSION.index + 1)
    OCR_SUBMISSION_PATH = SUBMISSION_PATH.with_name(
        SUBMISSION_PATH.name.replace(f"_conf{EVAL_CONF:g}", f"_ocr_conf{EVAL_CONF:g}")
    )
    OCR_SUBMISSION.to_csv(OCR_SUBMISSION_PATH, index=False, encoding="utf-8")
    test_reranked.to_csv(OCR_DIR / "test_reranked.csv", index=False)
    n_switched = int(test_reranked["ocr_switched"].sum())
    print(f"OCR로 클래스를 바꾼 test 박스 {n_switched}/{len(SUBMISSION)}개")
    print(f"OCR 제출 파일: {OCR_SUBMISSION_PATH}")
else:
    print("USE_OCR=False: OCR 제출 csv를 만들지 않습니다.")

EDA의 그림·캐시·보정 후보와 실제 적용 라벨을 구분합니다. train에만 있는 희귀 클래스의
AP는 검증할 수 없으며, 전체 56종의 일반화 성능을 측정했다고 해석하지 않습니다.
검증에 없는 클래스 수와 클래스별 AP를 단일 holdout 결과에 남깁니다.

참고: [Ultralytics 검증](https://docs.ultralytics.com/modes/val/),
[후처리·신뢰도·정밀도 설정](https://docs.ultralytics.com/usage/cfg/).
실행한 라이브러리 버전과 모든 명시 설정은 experiment.json 및 train/args.yaml에 기록됩니다.

## 9. Colab 런타임 삭제 (Colab에서만 실행)

제출 csv까지 만들었으면 Drive에 결과 쓰기를 마치고 런타임을 삭제합니다. 결과는 모두 Drive 실험 폴더에
있습니다. 셀 출력은 브라우저 탭에만 남으니 필요하면 **파일 → Drive에 사본 저장**으로 노트북을 저장합니다.
smoke 실행, 로컬, `COLAB_RELEASE_RUNTIME = False`이면 아무것도 하지 않습니다.


In [ ]:
if IN_COLAB and COLAB_RELEASE_RUNTIME and not SMOKE:
    if not SUBMISSION_PATH.is_file():
        raise FileNotFoundError(f"제출 csv가 없어 런타임을 남겨 둡니다: {SUBMISSION_PATH}")
    if USE_OCR and not OCR_SUBMISSION_PATH.is_file():
        raise FileNotFoundError(f"OCR 제출 csv가 없어 런타임을 남겨 둡니다: {OCR_SUBMISSION_PATH}")
    from google.colab import drive, runtime

    print(f"결과 폴더: {EXPERIMENT_DIR}")
    print("Drive에 쓰기를 마치고 Colab 런타임을 삭제합니다.")
    drive.flush_and_unmount()  # 끊기 전에 Drive에 아직 안 올라간 파일을 마저 올린다.
    runtime.unassign()
else:
    print("런타임을 삭제하지 않습니다(Colab 아님·smoke·COLAB_RELEASE_RUNTIME=False).")